# Main 2015 Retraining Notebook (Rigorous CV + Stacked Optimization)

This notebook rebuilds the full training workflow with stronger optimization rigor:

- 8 models: Logistic Regression, KNN, Naive Bayes, Random Forest, XGBoost, CatBoost, AdaBoost, LightGBM
- Feature engineering, KNN imputation, standardization, and collinearity filtering (cutoff = 0.70)
- Leakage guard for BP-derived target (drops BP-like features when target can be BP-defined)
- Two-stage CV optimization and final training:
  - Stage 1: 5-fold CV, broad random search, proxy epochs = 25
  - Stage 2: 5-fold CV, local refinement + additional exploration, proxy epochs = 80
  - Final: candidate re-competition on validation, proxy epochs = 300 with early stopping where supported
- Calibration: Isotonic, Platt Scaling, Venn-Abers
- Calibration metrics: Expected Calibration Error (ECE) and Log Loss
- Stacking: OOF feature generation + meta-learner optimization + weighted-blend search
- Explainability: SHAP and LIME

In [1]:
# Optional one-time installs (uncomment if needed), then restart kernel.
# %pip install -q numpy pandas scipy scikit-learn matplotlib seaborn joblib xgboost lightgbm catboost shap lime venn-abers

In [2]:
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import loguniform, randint, uniform

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

xgb_available = True
lgb_available = True
cat_available = True
shap_available = True
lime_available = True
venn_available = True

try:
    from xgboost import XGBClassifier
except Exception:
    xgb_available = False
    XGBClassifier = None

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except Exception:
    lgb_available = False
    lgb = None
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    cat_available = False
    CatBoostClassifier = None

try:
    import shap
except Exception:
    shap_available = False
    shap = None

try:
    from lime.lime_tabular import LimeTabularExplainer
except Exception:
    lime_available = False
    LimeTabularExplainer = None

try:
    from venn_abers import VennAbers
except Exception:
    venn_available = False
    VennAbers = None

import joblib

print({
    "xgboost": xgb_available,
    "lightgbm": lgb_available,
    "catboost": cat_available,
    "shap": shap_available,
    "lime": lime_available,
    "venn_abers": venn_available,
})

{'xgboost': True, 'lightgbm': True, 'catboost': True, 'shap': True, 'lime': True, 'venn_abers': True}


In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "main_2015_retraining_artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
EXPLAIN_DIR = ARTIFACT_DIR / "explanations"
LIME_DIR = EXPLAIN_DIR / "lime"

for p in [ARTIFACT_DIR, MODEL_DIR, EXPLAIN_DIR, LIME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    PROJECT_ROOT / "merged_clinical_dietary_leftjoin.csv",
    PROJECT_ROOT / "merged_clinical_leftjoin.csv",
    PROJECT_ROOT / "Datasets" / "Clinical" / "clinical.csv",
]

TARGET_CANDIDATES = ["hypertension", "htn", "target", "label", "outcome"]
ID_KEYWORDS = ["id", "subject", "participant", "seqn"]

COLLINEARITY_CUTOFF = 0.70
STAGE1_EPOCHS = 40
STAGE2_EPOCHS = 120
FINAL_EPOCHS = 500

STAGE1_TRIALS_PER_MODEL = 140
STAGE2_REFINEMENTS_PER_TOP_CONFIG = 28
TOP_K_STAGE1 = 10
TOP_K_STAGE2 = 4

CV_FOLDS_STAGE1 = 5
CV_FOLDS_STAGE2 = 6
STACK_OOF_FOLDS = 6
STACK_META_TRIALS = 400
STACK_BLEND_TRIALS = 7000
STACK_TOP_BASE_MODELS = 7

print(f"Artifacts: {ARTIFACT_DIR}")
print(
    {
        "stage1_trials_per_model": STAGE1_TRIALS_PER_MODEL,
        "stage2_refinements_per_top_config": STAGE2_REFINEMENTS_PER_TOP_CONFIG,
        "cv_folds": {"stage1": CV_FOLDS_STAGE1, "stage2": CV_FOLDS_STAGE2, "stack_oof": STACK_OOF_FOLDS},
        "stack_trials": {"meta": STACK_META_TRIALS, "blend": STACK_BLEND_TRIALS},
    }
)

Artifacts: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts
{'stage1_trials_per_model': 140, 'stage2_refinements_per_top_config': 28, 'cv_folds': {'stage1': 5, 'stage2': 6, 'stack_oof': 6}, 'stack_trials': {'meta': 400, 'blend': 7000}}


In [ ]:
def resolve_data_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No dataset found. Checked: {candidates}")


def infer_target_column(df, target_candidates):
    lc = {c.lower(): c for c in df.columns}
    for t in target_candidates:
        if t.lower() in lc:
            return lc[t.lower()]

    # Fallback: partial match on candidate tokens.
    for c in df.columns:
        c_lc = c.lower()
        if any(t.lower() in c_lc for t in target_candidates):
            return c
    return None


def infer_bp_columns(df):
    sbp_aliases = ["ave_sbp", "sbp", "systolic", "sysbp", "s_bp"]
    dbp_aliases = ["ave_dbp", "dbp", "diastolic", "diabp", "d_bp"]

    lc = {c.lower(): c for c in df.columns}

    sbp_col = None
    for a in sbp_aliases:
        if a in lc:
            sbp_col = lc[a]
            break
    if sbp_col is None:
        for c in df.columns:
            c_lc = c.lower()
            if any(a in c_lc for a in sbp_aliases):
                sbp_col = c
                break

    dbp_col = None
    for a in dbp_aliases:
        if a in lc:
            dbp_col = lc[a]
            break
    if dbp_col is None:
        for c in df.columns:
            c_lc = c.lower()
            if any(a in c_lc for a in dbp_aliases):
                dbp_col = c
                break

    return sbp_col, dbp_col


def find_bp_like_columns(columns):
    bp_tokens = [
        "ave_sbp",
        "ave_dbp",
        "sbp",
        "dbp",
        "systolic",
        "diastolic",
        "sysbp",
        "diabp",
        "blood_pressure",
    ]
    cols = []
    for c in columns:
        c_lc = c.lower()
        if any(tok in c_lc for tok in bp_tokens):
            cols.append(c)
    return sorted(set(cols))


def find_dictionary_files(candidate_dirs):
    paths = []
    for d in candidate_dirs:
        if d.exists():
            paths.extend(d.rglob("*data-dictionary*.csv"))
            paths.extend(d.rglob("*data_dictionary*.csv"))

    uniq = []
    seen = set()
    for p in paths:
        key = str(p.resolve()).lower()
        if key not in seen:
            seen.add(key)
            uniq.append(p)
    return sorted(uniq)


def extract_identifier_variables_from_dictionaries(dict_paths):
    explicit_name_tokens = {
        "hhnum",
        "member_code",
        "seqn",
        "respondent_id",
        "participant_id",
        "subject_id",
        "person_id",
        "record_id",
        "case_id",
        "household_id",
        "hhid",
        "hh_id",
    }
    label_tokens = [
        "unique identifier",
        "unique household identifier",
        "merging variable",
        "household number",
        "member code",
        "respondent identifier",
        "participant identifier",
        "subject identifier",
        "person identifier",
        "record identifier",
        "case identifier",
    ]

    identifier_vars = set()
    for p in dict_paths:
        try:
            ddf = pd.read_csv(p, dtype=str, low_memory=False)
        except Exception:
            continue

        col_map = {c.lower(): c for c in ddf.columns}
        var_col = col_map.get("variable_name")
        if var_col is None:
            for cand in ["variable", "var_name", "varname", "column", "column_name"]:
                if cand in col_map:
                    var_col = col_map[cand]
                    break

        label_col = None
        for cand in ["variable_label", "var_label", "label", "description", "value_label", "definition"]:
            if cand in col_map:
                label_col = col_map[cand]
                break

        if var_col is None:
            continue

        vars_series = ddf[var_col].fillna("").astype(str).str.strip()
        if label_col is not None:
            labels_series = ddf[label_col].fillna("").astype(str).str.strip()
        else:
            labels_series = pd.Series([""] * len(vars_series), index=vars_series.index)

        for var_name, var_label in zip(vars_series, labels_series):
            if not var_name or var_name.lower() == "nan":
                continue

            v_lc = var_name.lower()
            l_lc = var_label.lower()
            name_is_id = (
                v_lc in explicit_name_tokens
                or v_lc == "id"
                or v_lc.startswith("id_")
                or v_lc.endswith("_id")
            )
            label_is_id = any(tok in l_lc for tok in label_tokens)
            if name_is_id or label_is_id:
                identifier_vars.add(var_name)

    return sorted(identifier_vars)


def find_first_column_case_insensitive(columns, candidates):
    lc = {c.lower(): c for c in columns}
    for cand in candidates:
        cand_lc = cand.lower()
        if cand_lc in lc:
            return lc[cand_lc]
    for c in columns:
        c_lc = c.lower()
        if any(cand.lower() in c_lc for cand in candidates):
            return c
    return None


def to_numeric_clean(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.where(~s.isin([9, 99, 888888, 999999]), np.nan)


def build_smoking_level_feature(df_in):
    used_cols = []

    smoking_level_col = find_first_column_case_insensitive(df_in.columns, ["smoking_level"])
    smoke_status_col = find_first_column_case_insensitive(df_in.columns, ["smoke_status"])
    current_smoking_col = find_first_column_case_insensitive(df_in.columns, ["current_smoking", "currentsmoking"])
    ever_smoke_col = find_first_column_case_insensitive(df_in.columns, ["ever_smk"])

    if smoking_level_col is not None:
        s = to_numeric_clean(df_in[smoking_level_col]).clip(lower=0, upper=3)
        used_cols.append(smoking_level_col)
        return s.astype(float), sorted(set(used_cols))

    idx = df_in.index
    smoke = pd.Series(np.nan, index=idx, dtype=float)

    if smoke_status_col is not None:
        status = to_numeric_clean(df_in[smoke_status_col])
        used_cols.append(smoke_status_col)

        smoke.loc[status == 0] = 0
        smoke.loc[status == 2] = 1
        smoke.loc[status == 1] = 2

        if current_smoking_col is not None:
            current = to_numeric_clean(df_in[current_smoking_col])
            used_cols.append(current_smoking_col)
            smoke.loc[(status == 1) & (current == 3)] = 3

        return smoke.astype(float), sorted(set(used_cols))

    if current_smoking_col is not None:
        current = to_numeric_clean(df_in[current_smoking_col])
        used_cols.append(current_smoking_col)

        smoke.loc[current == 0] = 0
        smoke.loc[current.isin([1, 2])] = 2
        smoke.loc[current == 3] = 3

        if ever_smoke_col is not None:
            ever = to_numeric_clean(df_in[ever_smoke_col])
            used_cols.append(ever_smoke_col)
            smoke.loc[(current == 0) & (ever > 0)] = 1

        return smoke.astype(float), sorted(set(used_cols))

    if ever_smoke_col is not None:
        ever = to_numeric_clean(df_in[ever_smoke_col])
        used_cols.append(ever_smoke_col)
        smoke.loc[ever == 0] = 0
        smoke.loc[ever > 0] = 1
        return smoke.astype(float), sorted(set(used_cols))

    return None, []


def build_alcohol_level_feature(df_in):
    used_cols = []

    alcohol_level_col = find_first_column_case_insensitive(df_in.columns, ["alcohol_level"])
    alcohol_status_col = find_first_column_case_insensitive(df_in.columns, ["alcohol_status"])
    alcohol_ever_col = find_first_column_case_insensitive(df_in.columns, ["alcohol"])
    current_alcohol_col = find_first_column_case_insensitive(df_in.columns, ["con_alcohol"])
    drink30_col = find_first_column_case_insensitive(df_in.columns, ["drnk_30days"])
    binge_col = find_first_column_case_insensitive(df_in.columns, ["binge_drink", "binge_drinking"])

    if alcohol_level_col is not None:
        a = to_numeric_clean(df_in[alcohol_level_col]).clip(lower=0, upper=3)
        used_cols.append(alcohol_level_col)
        return a.astype(float), sorted(set(used_cols))

    idx = df_in.index
    alcohol = pd.Series(np.nan, index=idx, dtype=float)

    if alcohol_status_col is not None:
        status = to_numeric_clean(df_in[alcohol_status_col])
        used_cols.append(alcohol_status_col)

        alcohol.loc[status == 0] = 0
        alcohol.loc[status == 2] = 1
        alcohol.loc[status == 1] = 2

        if binge_col is not None:
            binge = to_numeric_clean(df_in[binge_col])
            used_cols.append(binge_col)
            alcohol.loc[(status == 1) & (binge == 1)] = 3

        return alcohol.astype(float), sorted(set(used_cols))

    alcohol.loc[:] = 0

    if alcohol_ever_col is not None:
        ever = to_numeric_clean(df_in[alcohol_ever_col])
        used_cols.append(alcohol_ever_col)
        alcohol.loc[ever > 0] = 1

    if current_alcohol_col is not None:
        current = to_numeric_clean(df_in[current_alcohol_col])
        used_cols.append(current_alcohol_col)
        alcohol.loc[current == 1] = np.maximum(alcohol.loc[current == 1], 2)

    if drink30_col is not None:
        d30 = to_numeric_clean(df_in[drink30_col])
        used_cols.append(drink30_col)
        alcohol.loc[d30 == 1] = np.maximum(alcohol.loc[d30 == 1], 2)

    if binge_col is not None:
        binge = to_numeric_clean(df_in[binge_col])
        used_cols.append(binge_col)
        alcohol.loc[binge == 1] = 3

    if used_cols:
        alcohol = alcohol.astype(float)
        alcohol.loc[~alcohol.index.isin(alcohol.dropna().index)] = np.nan
        return alcohol, sorted(set(used_cols))

    return None, []


def build_bmi_feature(df_in):
    weight_col = find_first_column_case_insensitive(df_in.columns, ["weight"])
    height_col = find_first_column_case_insensitive(df_in.columns, ["height"])

    if weight_col is None or height_col is None:
        return None, []

    w = pd.to_numeric(df_in[weight_col], errors="coerce")
    h = pd.to_numeric(df_in[height_col], errors="coerce")

    h_m = h.copy()
    if pd.notna(h_m.median(skipna=True)) and float(h_m.median(skipna=True)) > 3.0:
        h_m = h_m / 100.0

    bmi = w / (h_m ** 2)
    bmi = bmi.replace([np.inf, -np.inf], np.nan)

    return bmi.astype(float), [weight_col, height_col]


def build_whr_feature(df_in):
    waist_col = find_first_column_case_insensitive(df_in.columns, ["waist"])
    hip_col = find_first_column_case_insensitive(df_in.columns, ["hip"])

    if waist_col is None or hip_col is None:
        return None, []

    waist = pd.to_numeric(df_in[waist_col], errors="coerce")
    hip = pd.to_numeric(df_in[hip_col], errors="coerce").replace(0, np.nan)

    whr = (waist / hip).replace([np.inf, -np.inf], np.nan)
    return whr.astype(float), [waist_col, hip_col]


data_path = resolve_data_path(DATA_CANDIDATES)
df = pd.read_csv(data_path)
print(f"Loaded data: {data_path}")
print(f"Shape: {df.shape}")

target_col = infer_target_column(df, TARGET_CANDIDATES)
TARGET_DEFINED_FROM_BP = False
TARGET_SOURCE_COLUMNS = []

if target_col is None:
    sbp_col, dbp_col = infer_bp_columns(df)
    if sbp_col is not None and dbp_col is not None:
        sbp = pd.to_numeric(df[sbp_col], errors="coerce")
        dbp = pd.to_numeric(df[dbp_col], errors="coerce")
        df["Hypertension"] = (((sbp >= 130) | (dbp >= 80)).fillna(False)).astype(int)
        target_col = "Hypertension"
        TARGET_DEFINED_FROM_BP = True
        TARGET_SOURCE_COLUMNS = [sbp_col, dbp_col]
        print(f"Target column created from: {sbp_col}, {dbp_col}")
    else:
        raise ValueError(
            f"Could not infer target column from {TARGET_CANDIDATES} and could not derive from BP columns. "
            f"Found BP-like columns: {[c for c in df.columns if 'bp' in c.lower()][:20]}"
        )

df = df.dropna(subset=[target_col]).copy()
y_raw = df[target_col]

if y_raw.nunique() != 2:
    raise ValueError(f"Target must be binary. Found {y_raw.nunique()} classes in '{target_col}'.")

if y_raw.dtype == "O":
    y = pd.Series(LabelEncoder().fit_transform(y_raw.astype(str)), index=y_raw.index, name=target_col)
else:
    y = pd.Series(y_raw.astype(int), index=y_raw.index, name=target_col)

X = df.drop(columns=[target_col]).copy()
RAW_VARIABLES_BEFORE_CULL = sorted(X.columns.tolist())
print(f"\nVariables before culling: {len(RAW_VARIABLES_BEFORE_CULL)}")
for v in RAW_VARIABLES_BEFORE_CULL:
    print(v)

# Engineer smoking and alcohol features separately.
ENGINEERED_BASE_FEATURES_CREATED = []
ENGINEERED_BASE_FEATURE_SOURCE_MAP = {}

smoking_feature, smoking_sources = build_smoking_level_feature(X)
if smoking_feature is not None:
    X["fe_smoking_level"] = smoking_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("fe_smoking_level")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["fe_smoking_level"] = smoking_sources

alcohol_feature, alcohol_sources = build_alcohol_level_feature(X)
if alcohol_feature is not None:
    X["fe_alcohol_level"] = alcohol_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("fe_alcohol_level")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["fe_alcohol_level"] = alcohol_sources

# Engineer BMI and WHR, then remove source anthropometrics.
bmi_feature, bmi_source_cols = build_bmi_feature(X)
if bmi_feature is not None:
    X["bmi"] = bmi_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("bmi")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["bmi"] = bmi_source_cols

whr_feature, whr_source_cols = build_whr_feature(X)
if whr_feature is not None:
    X["whr"] = whr_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("whr")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["whr"] = whr_source_cols

print(f"\nEngineered base features created: {ENGINEERED_BASE_FEATURES_CREATED}")
print(f"Engineered base feature sources: {ENGINEERED_BASE_FEATURE_SOURCE_MAP}")

BEHAVIOR_RAW_CANDIDATES = [
    "current_smoking",
    "currentsmoking",
    "ever_smk",
    "smoke_status",
    "smoking_level",
    "alcohol",
    "con_alcohol",
    "drnk_30days",
    "drnk_30d_num",
    "alcohol_status",
    "binge_drink",
    "binge_drinking",
    "alcohol_level",
]
x_lc = {c.lower(): c for c in X.columns}
BEHAVIOR_RAW_PURGED_COLUMNS = sorted({x_lc[c.lower()] for c in BEHAVIOR_RAW_CANDIDATES if c.lower() in x_lc})
if BEHAVIOR_RAW_PURGED_COLUMNS:
    X = X.drop(columns=BEHAVIOR_RAW_PURGED_COLUMNS, errors="ignore")
print(f"Purged smoking/alcohol raw columns: {BEHAVIOR_RAW_PURGED_COLUMNS}")

ANTHRO_SOURCE_DROPPED_COLUMNS = []
anthro_source_cols = sorted(set((bmi_source_cols or []) + (whr_source_cols or [])))
if anthro_source_cols:
    ANTHRO_SOURCE_DROPPED_COLUMNS = sorted({c for c in anthro_source_cols if c in X.columns and c.lower() not in {"bmi", "whr"}})
    if ANTHRO_SOURCE_DROPPED_COLUMNS:
        X = X.drop(columns=ANTHRO_SOURCE_DROPPED_COLUMNS, errors="ignore")

# Backward-compatible variable used by downstream audit cells.
HEIGHT_WEIGHT_DROPPED_COLUMNS = ANTHRO_SOURCE_DROPPED_COLUMNS.copy()
print(f"Dropped anthropometric source columns after BMI/WHR engineering: {HEIGHT_WEIGHT_DROPPED_COLUMNS}")

# Hard-drop known survey design and administrative columns.
MANUAL_NON_PREDICTIVE_CANDIDATES = [
    "regcode",
    "provcode",
    "provhuc",
    "psc",
    "csc",
    "rhc",
    "psurec",
    "strrec",
    "wgts",
    "fwgt",
    "finalwgt",
    "finalwgt1",
    "finalwgt4",
    "fwgth_natl_var",
    "fwgth_prov",
    "fwgth_natl2_var",
    "fwgti_natl_var",
    "fwgti_prov",
    "fwgti_natl2_var",
    "fwgti_prov2",
    "rep_natl",
    "rep_prov",
    "ms_psucode",
    "enns_year",
    "wrkplace",
    "interview_status",
    "intdate",
    "enumcode",
]
x_lc = {c.lower(): c for c in X.columns}
MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS = sorted({x_lc[c.lower()] for c in MANUAL_NON_PREDICTIVE_CANDIDATES if c.lower() in x_lc})
if MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS:
    X = X.drop(columns=MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS, errors="ignore")
print(f"Dropped manual non-predictive columns: {MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS}")

# Leakage guard: if target is BP-derived (or likely BP-defined), remove BP-like features from X.
BP_LEAKAGE_GUARD_ACTIVE = TARGET_DEFINED_FROM_BP or (target_col.lower() in {"hypertension", "htn"})
BP_DROPPED_COLUMNS = []
if BP_LEAKAGE_GUARD_ACTIVE:
    BP_DROPPED_COLUMNS = find_bp_like_columns(X.columns)
    BP_DROPPED_COLUMNS = sorted(set(BP_DROPPED_COLUMNS + TARGET_SOURCE_COLUMNS))
    X = X.drop(columns=BP_DROPPED_COLUMNS, errors="ignore")
    print(f"Dropped BP-like columns to prevent leakage: {BP_DROPPED_COLUMNS}")

# Dictionary-driven identifier drop (before split and before collinearity checks).
DICT_DIR_CANDIDATES = [
    PROJECT_ROOT / "Datasets",
    PROJECT_ROOT / "Datasets2015",
    PROJECT_ROOT.parent / "Datasets",
    PROJECT_ROOT.parent / "Datasets2015",
]
DICTIONARY_FILES_USED = find_dictionary_files(DICT_DIR_CANDIDATES)
DICT_IDENTIFIER_VARIABLES = extract_identifier_variables_from_dictionaries(DICTIONARY_FILES_USED)

x_lc = {c.lower(): c for c in X.columns}
DICT_IDENTIFIER_DROPPED_COLUMNS = sorted({x_lc[v.lower()] for v in DICT_IDENTIFIER_VARIABLES if v.lower() in x_lc})
if DICT_IDENTIFIER_DROPPED_COLUMNS:
    X = X.drop(columns=DICT_IDENTIFIER_DROPPED_COLUMNS, errors="ignore")
    print(f"Dropped dictionary-identified ID columns: {DICT_IDENTIFIER_DROPPED_COLUMNS}")
else:
    print("Dropped dictionary-identified ID columns: []")
print(f"Dictionary files scanned: {len(DICTIONARY_FILES_USED)}")

VARIABLES_AFTER_BASE_CULL = sorted(X.columns.tolist())
print(f"\nVariables after culling (before split): {len(VARIABLES_AFTER_BASE_CULL)}")
for v in VARIABLES_AFTER_BASE_CULL:
    print(v)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_tmp,
)

# Additional train-only fallback for ID/high-cardinality drops.
drop_id_like = []
for c in X_train.columns:
    c_lc = c.lower()
    if any(k in c_lc for k in ID_KEYWORDS):
        drop_id_like.append(c)
        continue
    nunique_ratio = X_train[c].nunique(dropna=False) / max(len(X_train), 1)
    if nunique_ratio > 0.995:
        drop_id_like.append(c)

X_train = X_train.drop(columns=drop_id_like, errors="ignore")
X_valid = X_valid.drop(columns=drop_id_like, errors="ignore")
X_test = X_test.drop(columns=drop_id_like, errors="ignore")

print(f"\nTarget column: {target_col}")
print(f"Dropped behavior raw columns: {len(BEHAVIOR_RAW_PURGED_COLUMNS)}")
print(f"Dropped anthropometric source columns: {len(HEIGHT_WEIGHT_DROPPED_COLUMNS)}")
print(f"Dropped manual non-predictive columns: {len(MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS)}")
print(f"Dropped dictionary ID columns: {len(DICT_IDENTIFIER_DROPPED_COLUMNS)}")
print(f"Dropped ID-like columns (train-only fallback): {len(drop_id_like)}")
print(f"Train/Valid/Test: {X_train.shape}, {X_valid.shape}, {X_test.shape}")

Loaded data: c:\Jon\College\Thesis\V2.2.1.1\merged_clinical_dietary_leftjoin.csv
Shape: (151189, 63)
Target column created from: Ave_SBP, Ave_DBP

Variables before culling: 63
Ave_DBP
Ave_SBP
Total_CHO
Total_Calcium
Total_Energy
Total_Fat
Total_FoodIntake
Total_Iron
Total_Niacin
Total_Protein
Total_Riboflavin
Total_Thiamin
Total_VitaminA
Total_VitaminC
age
alcohol
alcohol_status
binge_drinking
con_alcohol
csc
cu
current_smoking
drnk_30d_num
drnk_30days
ever_smk
fg1
fg10
fg11
fg12
fg13
fg14
fg15
fg16
fg17
fg18
fg19
fg2
fg20
fg21
fg23
fg24
fg25
fg26
fg27
fg3
fg4
fg5
fg6
fg7
fg8
fg9
finalwgt1
finalwgt4
hhnum
member_code
provcode
psc
psurec
regcode
sex
smoke_status
strrec
wgts

Engineered base features created: ['fe_smoking_level', 'fe_alcohol_level']
Engineered base feature sources: {'fe_smoking_level': ['current_smoking', 'smoke_status'], 'fe_alcohol_level': ['alcohol_status', 'binge_drinking']}
Purged smoking/alcohol raw columns: ['alcohol', 'alcohol_status', 'binge_drinking', 'con_alco

In [5]:
# Fast raw-variable audit (before feature engineering/collinearity).
def _name_identifier_like(col_name):
    c = str(col_name).lower()
    tokens = ["id", "subject", "participant", "seqn", "identifier", "record", "uuid", "case", "sample", "hhnum", "member_code"]
    return any(t in c for t in tokens)


raw_vars = [c for c in df.columns if c != target_col]
raw_var_audit_df = pd.DataFrame({"variable": raw_vars})
raw_var_audit_df["identifier_like_name"] = raw_var_audit_df["variable"].apply(_name_identifier_like)
raw_var_audit_df["dropped_bp_guard"] = raw_var_audit_df["variable"].isin(BP_DROPPED_COLUMNS)
raw_var_audit_df["dropped_dict_rule"] = raw_var_audit_df["variable"].isin(globals().get("DICT_IDENTIFIER_DROPPED_COLUMNS", []))
raw_var_audit_df["dropped_manual_rule"] = raw_var_audit_df["variable"].isin(globals().get("MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS", []))
raw_var_audit_df["dropped_behavior_purge"] = raw_var_audit_df["variable"].isin(globals().get("BEHAVIOR_RAW_PURGED_COLUMNS", []))
raw_var_audit_df["dropped_height_weight_rule"] = raw_var_audit_df["variable"].isin(globals().get("HEIGHT_WEIGHT_DROPPED_COLUMNS", []))
raw_var_audit_df["dropped_fallback_id_rule"] = raw_var_audit_df["variable"].isin(drop_id_like)
raw_var_audit_df["kept_after_base_filters"] = raw_var_audit_df["variable"].isin(X_train.columns)

kept_identifier_like_df = raw_var_audit_df[
    (raw_var_audit_df["identifier_like_name"])
    & (raw_var_audit_df["kept_after_base_filters"])
] .copy()

raw_var_audit_path = ARTIFACT_DIR / "raw_variable_drop_audit_2015_rebuild.csv"
kept_identifier_like_path = ARTIFACT_DIR / "raw_variable_kept_identifier_like_2015_rebuild.csv"
raw_var_audit_df.to_csv(raw_var_audit_path, index=False)
kept_identifier_like_df.to_csv(kept_identifier_like_path, index=False)

print(f"Saved: {raw_var_audit_path}")
print(f"Saved: {kept_identifier_like_path}")
print("Engineered base features:", globals().get("ENGINEERED_BASE_FEATURES_CREATED", []))
print("Drop summary:")
print(
    {
        "raw_variables": int(len(raw_var_audit_df)),
        "dropped_bp_guard": int(raw_var_audit_df['dropped_bp_guard'].sum()),
        "dropped_dict_rule": int(raw_var_audit_df['dropped_dict_rule'].sum()),
        "dropped_manual_rule": int(raw_var_audit_df['dropped_manual_rule'].sum()),
        "dropped_behavior_purge": int(raw_var_audit_df['dropped_behavior_purge'].sum()),
        "dropped_height_weight_rule": int(raw_var_audit_df['dropped_height_weight_rule'].sum()),
        "dropped_fallback_id_rule": int(raw_var_audit_df['dropped_fallback_id_rule'].sum()),
        "kept_identifier_like": int(len(kept_identifier_like_df)),
    }
)
kept_identifier_like_df.head(30)

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\raw_variable_drop_audit_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\raw_variable_kept_identifier_like_2015_rebuild.csv
Engineered base features: ['fe_smoking_level', 'fe_alcohol_level']
Drop summary:
{'raw_variables': 63, 'dropped_bp_guard': 2, 'dropped_dict_rule': 2, 'dropped_manual_rule': 9, 'dropped_behavior_purge': 9, 'dropped_height_weight_rule': 0, 'dropped_fallback_id_rule': 0, 'kept_identifier_like': 0}


,variable,identifier_like_name,dropped_bp_guard,dropped_dict_rule,dropped_manual_rule,dropped_behavior_purge,dropped_height_weight_rule,dropped_fallback_id_rule,kept_after_base_filters


In [6]:
def find_first_column(cols, keys):
    low = {c.lower(): c for c in cols}
    for k in keys:
        for c_lc, c in low.items():
            if k in c_lc:
                return c
    return None


def feature_engineering(df_in):
    df_fe = df_in.copy()
    engineered_added = []

    age_col = find_first_column(df_fe.columns, ["age", "agemos"])
    bmi_col = find_first_column(df_fe.columns, ["bmi"])
    chol_col = find_first_column(df_fe.columns, ["chol"])
    hdl_col = find_first_column(df_fe.columns, ["hdl"])
    waist_col = find_first_column(df_fe.columns, ["waist"])
    hip_col = find_first_column(df_fe.columns, ["hip"])
    energy_col = find_first_column(df_fe.columns, ["energy", "kcal"])
    fat_col = find_first_column(df_fe.columns, ["total_fat", "fat"])

    if age_col and bmi_col:
        df_fe["fe_age_x_bmi"] = df_fe[age_col] * df_fe[bmi_col]
        engineered_added.append("fe_age_x_bmi")

    if chol_col and hdl_col:
        den = df_fe[hdl_col].replace(0, np.nan)
        df_fe["fe_chol_hdl_ratio"] = df_fe[chol_col] / den
        engineered_added.append("fe_chol_hdl_ratio")

    if waist_col and hip_col:
        den = df_fe[hip_col].replace(0, np.nan)
        df_fe["fe_waist_hip_ratio"] = df_fe[waist_col] / den
        engineered_added.append("fe_waist_hip_ratio")

    if energy_col and fat_col:
        den = df_fe[energy_col].replace(0, np.nan)
        df_fe["fe_fat_density"] = df_fe[fat_col] / den
        engineered_added.append("fe_fat_density")

    return df_fe, sorted(set(engineered_added))


X_train_fe, engineered_train = feature_engineering(X_train)
X_valid_fe, engineered_valid = feature_engineering(X_valid)
X_test_fe, engineered_test = feature_engineering(X_test)

all_cols = sorted(set(X_train_fe.columns) | set(X_valid_fe.columns) | set(X_test_fe.columns))
X_train_fe = X_train_fe.reindex(columns=all_cols)
X_valid_fe = X_valid_fe.reindex(columns=all_cols)
X_test_fe = X_test_fe.reindex(columns=all_cols)

ENGINEERED_FEATURES_PHASE2 = sorted(set(engineered_train) | set(engineered_valid) | set(engineered_test))
print(f"Feature counts after engineering: {X_train_fe.shape[1]}")
print("Phase-2 engineered features:", ENGINEERED_FEATURES_PHASE2)

Feature counts after engineering: 44
Phase-2 engineered features: ['fe_fat_density']


In [7]:
X_train_fe.describe()

,Total_CHO,Total_Calcium,Total_Energy,Total_Fat,Total_FoodIntake,Total_Iron,Total_Niacin,Total_Protein,Total_Riboflavin,Total_Thiamin,Total_VitaminA,Total_VitaminC,age,cu,fe_alcohol_level,fe_fat_density,fe_smoking_level,fg1,fg10,fg11,fg12,fg13,fg14,fg15,fg16,fg17,fg18,fg19,fg2,fg20,fg21,fg23,fg24,fg25,fg26,fg27,fg3,fg4,fg5,fg6,fg7,fg8,fg9,sex
count,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,105832.000000,26632.000000,21439.000000,26632.000000,21508.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,105832.000000
mean,1607.702395,45.436596,9050.155086,170.502139,4021.191006,1964.297185,89.407654,268.304206,3.340063,3.989993,2090.041617,216.523177,30.792238,5.368160,0.915901,0.018501,0.592989,1803.295577,407.913482,170.295439,26.130729,144.164710,808.695617,451.178002,242.319226,115.198389,78.579381,179.361573,1556.335385,144.974132,34.387441,67.912263,122.196366,71.163833,44.622193,6.410340,136.164631,110.795560,70.551733,58.387921,39.820271,622.094864,214.181382,1.517131
std,863.339120,26.781527,4590.014890,152.818796,2303.951065,1860.025292,52.305876,145.542185,2.680020,2.824709,5225.308101,269.486528,20.815613,2.423017,1.086676,0.011287,1.065638,1014.161686,701.885898,736.188197,153.078234,717.004068,738.256291,535.477373,440.314320,314.539900,121.365095,547.604845,1010.845136,413.182077,342.543374,128.741090,197.023447,172.986236,58.724649,58.829078,520.089468,173.521919,338.215809,96.458694,90.123325,853.668369,399.135141,0.499709
min,32.910000,1.190020,292.459991,0.581470,79.649994,48.730000,3.286000,7.009060,0.065510,0.082105,0.000000,0.000000,3.010000,0.500000,0.000000,0.001250,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,1022.560059,28.129999,5942.060059,70.220001,2582.050049,1075.177551,54.182190,171.850281,1.910019,2.220000,642.399963,54.270000,12.510000,4.000000,0.000000,0.009931,0.000000,1101.574982,35.400002,0.000000,0.000000,0.000000,300.070007,65.629997,0.000000,0.000000,0.000000,0.000000,900.700012,0.000000,0.000000,11.900000,33.799999,4.000000,11.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,118.400002,0.000000,1.000000
50%,1463.383057,40.009998,8284.234863,129.834549,3628.559937,1580.420044,79.307999,241.050003,2.787200,3.310000,1161.859985,143.020004,25.940000,5.000000,0.000000,0.016557,0.000000,1630.729981,205.100006,0.000000,0.000000,0.000000,647.099976,307.630005,0.000000,0.000000,0.000000,0.000000,1413.579956,0.000000,0.000000,38.280001,68.900002,25.100000,27.000000,0.000000,0.000000,55.000000,0.000000,30.100000,0.000000,408.700012,71.769997,2.000000
75%,2005.229858,56.970001,11265.429688,222.110001,4911.869629,2342.409912,111.092497,332.519989,4.002721,4.920000,1953.139984,284.175003,47.180000,7.000000,2.000000,0.025082,1.000000,2291.052612,562.928757,59.000000,0.000000,0.000000,1096.390015,641.380020,346.039978,0.000000,135.199997,170.222500,2057.500000,118.180000,0.000000,81.892500,129.224998,60.000000,56.825003,0.000000,0.000000,148.734997,0.000000,80.889999,40.299999,858.000000,276.000000,2.000000
max,13414.833008,433.129974,63586.351563,1923.150024,43814.109375,70840.976563,564.910034,2069.939941,54.241108,31.086248,146662.671880,7595.199707,100.710000,24.666667,3.000000,0.080167,3.000000,15839.702148,18890.189453,34014.738281,4164.160156,34014.738281,10857.070313,9472.160156,5769.790039,4586.700195,1139.500000,16728.750000,15494.560547,13601.000000,16500.000000,3660.199951,601

In [8]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


num_cols = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train_fe.columns if c not in num_cols]

numeric_pipe = Pipeline(
    steps=[
        ("imputer", KNNImputer(n_neighbors=5)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

X_train_proc = preprocessor.fit_transform(X_train_fe)
X_valid_proc = preprocessor.transform(X_valid_fe)
X_test_proc = preprocessor.transform(X_test_fe)

feature_names = preprocessor.get_feature_names_out()

X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train_fe.index)
X_valid_proc = pd.DataFrame(X_valid_proc, columns=feature_names, index=X_valid_fe.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=feature_names, index=X_test_fe.index)

COLLINEARITY_PROTECTED_FEATURES = sorted(
    [c for c in ["num__bmi", "num__whr"] if c in X_train_proc.columns]
)


def collinearity_filter(df_in, cutoff=0.70, protected_cols=None):
    protected = set(protected_cols or [])
    corr = df_in.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    drop_cols = []
    for c in upper.columns:
        if c in protected:
            continue
        if (upper[c] > cutoff).any():
            drop_cols.append(c)

    keep_cols = [c for c in df_in.columns if c not in drop_cols]
    return keep_cols, drop_cols


keep_cols, dropped_cols = collinearity_filter(
    X_train_proc,
    cutoff=COLLINEARITY_CUTOFF,
    protected_cols=COLLINEARITY_PROTECTED_FEATURES,
)

X_train_final = X_train_proc[keep_cols].copy()
X_valid_final = X_valid_proc[keep_cols].copy()
X_test_final = X_test_proc[keep_cols].copy()

if "BP_LEAKAGE_GUARD_ACTIVE" in globals() and BP_LEAKAGE_GUARD_ACTIVE:
    bp_tokens_guard = ["ave_sbp", "ave_dbp", "sbp", "dbp", "systolic", "diastolic", "sysbp", "diabp", "blood_pressure"]
    leaked_bp_cols = [c for c in X_train_final.columns if any(tok in c.lower() for tok in bp_tokens_guard)]
    if leaked_bp_cols:
        raise ValueError(f"Leakage guard failed: BP-like features still present in model matrix: {leaked_bp_cols}")
    print("BP leakage guard: passed (no BP-like features in final matrix)")

print(f"Processed feature count: {len(feature_names)}")
print(f"Protected from collinearity dropping: {COLLINEARITY_PROTECTED_FEATURES}")
print(f"Dropped for collinearity (>{COLLINEARITY_CUTOFF}): {len(dropped_cols)}")
print(f"Final feature count: {X_train_final.shape[1]}")

BP leakage guard: passed (no BP-like features in final matrix)
Processed feature count: 44
Protected from collinearity dropping: []
Dropped for collinearity (>0.7): 15
Final feature count: 29


In [9]:
# Sanity checks for requested variable cleanup and feature engineering.
requested_remove = ["provcode", "psc", "psurec", "regcode", "strrec", "wgts", "weight", "height", "hip", "waist"]

print("Manual non-predictive dropped columns:")
print(sorted(globals().get("MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS", [])))

print("\nBehavior raw columns purged:")
print(sorted(globals().get("BEHAVIOR_RAW_PURGED_COLUMNS", [])))

print("\nAnthropometric source columns dropped after BMI/WHR engineering:")
print(sorted(globals().get("HEIGHT_WEIGHT_DROPPED_COLUMNS", [])))

print("\nEngineered base features created:")
print(globals().get("ENGINEERED_BASE_FEATURES_CREATED", []))
print(globals().get("ENGINEERED_BASE_FEATURE_SOURCE_MAP", {}))

print("\nSeparate engineered behavior features present:")
print("  fe_smoking_level:", "fe_smoking_level" in X_train.columns)
print("  fe_alcohol_level:", "fe_alcohol_level" in X_train.columns)
print("  bmi:", "bmi" in X_train.columns)
print("  whr:", "whr" in X_train.columns)

fe_log1p_cols = [c for c in X_train_fe.columns if c.startswith("fe_log1p_")]
print("\nfe_log1p engineered columns count:", len(fe_log1p_cols))
print("Phase-2 engineered features:", globals().get("ENGINEERED_FEATURES_PHASE2", []))

print("\nCollinearity protected processed features:", globals().get("COLLINEARITY_PROTECTED_FEATURES", []))

print("\nRequested remove status in base train features:")
for c in requested_remove:
    print(f"  {c}:", c in X_train.columns)

final_lower = [c.lower() for c in X_train_final.columns]
print("\nRequested remove tokens in final model matrix names:")
for c in requested_remove:
    has_token = any(c in name for name in final_lower)
    print(f"  {c}:", has_token)

Manual non-predictive dropped columns:
['csc', 'finalwgt1', 'finalwgt4', 'provcode', 'psc', 'psurec', 'regcode', 'strrec', 'wgts']

Behavior raw columns purged:
['alcohol', 'alcohol_status', 'binge_drinking', 'con_alcohol', 'current_smoking', 'drnk_30d_num', 'drnk_30days', 'ever_smk', 'smoke_status']

Anthropometric source columns dropped after BMI/WHR engineering:
[]

Engineered base features created:
['fe_smoking_level', 'fe_alcohol_level']
{'fe_smoking_level': ['current_smoking', 'smoke_status'], 'fe_alcohol_level': ['alcohol_status', 'binge_drinking']}

Separate engineered behavior features present:
  fe_smoking_level: True
  fe_alcohol_level: True
  bmi: False
  whr: False

fe_log1p engineered columns count: 0
Phase-2 engineered features: ['fe_fat_density']

Collinearity protected processed features: []

Requested remove status in base train features:
  provcode: False
  psc: False
  psurec: False
  regcode: False
  strrec: False
  wgts: False
  weight: False
  height: False
  hip

In [10]:
# Diagnose anthropometric source availability for BMI/WHR engineering.
anthro_cols = [c for c in df.columns if any(tok in c.lower() for tok in ["weight", "height", "bmi", "waist", "hip", "whr"])]
print("Anthropometric columns found in raw merged dataframe:", anthro_cols)

x_train_cols = [c for c in X_train.columns if any(tok in c.lower() for tok in ["weight", "height", "bmi", "waist", "hip", "whr"])]
print("Anthropometric columns retained in X_train:", x_train_cols)

Anthropometric columns found in raw merged dataframe: []
Anthropometric columns retained in X_train: []


In [11]:
# Variable audit: raw -> engineered -> processed -> final model matrix.
def _is_identifier_like(name):
    n = str(name).lower()
    id_tokens = ["member_code", "hhnum"]
    return any(tok in n for tok in id_tokens)


def _is_bp_like(name):
    n = str(name).lower()
    bp_tokens = ["ave_sbp", "ave_dbp"]
    return any(tok in n for tok in bp_tokens)


dict_drop_set = set(globals().get("DICT_IDENTIFIER_DROPPED_COLUMNS", []))
base_predictor_cols = [c for c in df.columns if c != target_col]
train_base_before_id_drop = X.loc[y_train.index].copy()
nunique_ratio_map = {
    c: train_base_before_id_drop[c].nunique(dropna=False) / max(len(train_base_before_id_drop), 1)
    for c in train_base_before_id_drop.columns
}

audit_rows = []
for col in base_predictor_cols:
    in_bp_drop = col in BP_DROPPED_COLUMNS
    in_dict_drop = col in dict_drop_set
    in_id_drop = col in drop_id_like
    kept_after_base_filters = col in X_train.columns
    uniq_ratio = nunique_ratio_map.get(col, np.nan)

    review_bits = []
    if in_dict_drop:
        review_bits.append("dropped by dictionary identifier rule")
    if in_id_drop:
        review_bits.append("dropped by train-only fallback rule")
    if _is_identifier_like(col) and kept_after_base_filters and not in_dict_drop and not in_id_drop:
        review_bits.append("identifier-like retained in base features")
    if _is_bp_like(col) and kept_after_base_filters and BP_LEAKAGE_GUARD_ACTIVE:
        review_bits.append("bp-like retained while BP guard active")

    audit_rows.append(
        {
            "stage": "raw_base",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": uniq_ratio,
            "dropped_bp_guard": in_bp_drop,
            "dropped_dict_rule": in_dict_drop,
            "dropped_id_rule": in_id_drop,
            "kept_after_base_filters": kept_after_base_filters,
            "review_flag": "; ".join(review_bits),
        }
    )

engineered_only = sorted(set(X_train_fe.columns) - set(X_train.columns))
for col in engineered_only:
    audit_rows.append(
        {
            "stage": "engineered",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": col in X_train_fe.columns,
            "review_flag": "",
        }
    )

for col in feature_names:
    audit_rows.append(
        {
            "stage": "processed",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": col in keep_cols,
            "review_flag": "dropped by collinearity" if col in dropped_cols else "",
        }
    )

final_var = X_train_final.var(numeric_only=True)
for col in X_train_final.columns:
    near_zero_var = bool(final_var.get(col, np.nan) <= 1e-10) if pd.notna(final_var.get(col, np.nan)) else False
    review_bits = []
    if _is_identifier_like(col):
        review_bits.append("identifier-like feature in final matrix")
    if _is_bp_like(col) and BP_LEAKAGE_GUARD_ACTIVE:
        review_bits.append("bp-like feature in final matrix")
    if near_zero_var:
        review_bits.append("near-zero variance in final matrix")

    audit_rows.append(
        {
            "stage": "final_model",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": True,
            "review_flag": "; ".join(review_bits),
        }
    )

variable_audit_df = pd.DataFrame(audit_rows)
variable_audit_df = variable_audit_df.sort_values(["stage", "variable"]).reset_index(drop=True)

suspect_kept_df = variable_audit_df[
    (variable_audit_df["stage"] == "final_model")
    & (variable_audit_df["review_flag"].astype(str).str.len() > 0)
] .copy()

runtime_var_names = sorted([name for name in globals().keys() if not name.startswith("_")])
runtime_var_df = pd.DataFrame({"name": runtime_var_names})
runtime_var_df["likely_temporary"] = runtime_var_df["name"].apply(
    lambda n: (len(n) <= 2 and n.islower()) or n in {"rows", "params", "met", "i", "c", "m", "p"}
 )

var_audit_path = ARTIFACT_DIR / "variable_audit_2015_rebuild.csv"
suspect_path = ARTIFACT_DIR / "variable_audit_suspect_kept_2015_rebuild.csv"
runtime_path = ARTIFACT_DIR / "runtime_variable_list_2015_rebuild.csv"

variable_audit_df.to_csv(var_audit_path, index=False)
suspect_kept_df.to_csv(suspect_path, index=False)
runtime_var_df.to_csv(runtime_path, index=False)

print(f"Saved: {var_audit_path}")
print(f"Saved: {suspect_path}")
print(f"Saved: {runtime_path}")
print(f"Dictionary files scanned: {len(globals().get('DICTIONARY_FILES_USED', []))}")
print(f"Dictionary ID columns dropped: {sorted(list(dict_drop_set))}")
print("\nAudit counts by stage:")
print(variable_audit_df.groupby("stage")["variable"].count().to_string())
print("\nPotentially suspect kept final features:", len(suspect_kept_df))
suspect_kept_df.head(30)

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\variable_audit_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\variable_audit_suspect_kept_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\runtime_variable_list_2015_rebuild.csv
Dictionary files scanned: 10
Dictionary ID columns dropped: ['hhnum', 'member_code']

Audit counts by stage:
stage
engineered      1
final_model    29
processed      44
raw_base       63

Potentially suspect kept final features: 0


,stage,variable,identifier_like,bp_like,train_nunique_ratio,dropped_bp_guard,dropped_dict_rule,dropped_id_rule,kept_after_base_filters,review_flag


In [12]:
X_train_final

,num__Total_CHO,num__Total_Fat,num__Total_Iron,num__Total_Riboflavin,num__Total_VitaminC,num__age,num__cu,num__fe_alcohol_level,num__fe_smoking_level,num__fg10,num__fg11,num__fg12,num__fg15,num__fg16,num__fg17,num__fg18,num__fg19,num__fg21,num__fg23,num__fg24,num__fg26,num__fg27,num__fg3,num__fg4,num__fg5,num__fg6,num__fg7,num__fg9,num__sex
139444,-0.181571,1.339865,0.256495,0.177992,0.274761,0.387104,0.036456,0.017162,-0.650788,-0.097102,1.311482,6.329971,0.020521,-0.145057,-0.265213,0.063797,0.250122,0.151464,5.605932,3.370224,-0.978595,0.443538,-0.411930,1.284097,0.271838,-0.237783,0.359146,-0.374455,0.966305
40637,-0.031776,-0.030838,0.170406,-0.058773,0.634366,-0.916251,-0.336958,-0.715819,-0.650788,0.589375,-0.205038,-0.271790,-0.910534,-0.584077,0.184417,0.739421,0.980581,2.121927,0.045735,0.578249,0.562480,-0.171331,0.144754,0.133096,-0.296737,0.119615,1.803018,1.492237,0.966305
139758,0.244385,-1.021471,-0.301650,-0.067687,-0.278582,1.091866,0.036456,1.483125,0.660332,-0.146332,-0.357075,-0.271790,-0.403510,-0.861310,0.291384,-0.035598,-0.515326,-0.156272,-0.402797,-0.452024,-0.762726,0.090070,0.091662,-0.911450,-0.325200,-0.183942,-0.694466,0.144992,-1.034869
35179,0.100308,-0.250164,-0.294809,-0.132420,-0.933582,-0.519912,0.036456,-0.471492,-0.650788,-0.479429,-0.345915,-0.239920,-0.164245,-0.406774,-0.098750,1.439932,3.017425,5.143241,-0.321410,0.185125,0.320365,-0.171331,-0.408528,-0.228185,-0.198255,0.483223,-0.167239,-0.818818,0.966305
19784,-1.753054,-1.155440,-1.030506,-1.227997,-1.260691,-1.024344,-0.834843,-0.960146,-0.650788,-0.906100,-0.357075,-0.271790,-0.671698,0.041205,-0.574047,-1.012503,-0.515326,-0.156272,-0.666829,-0.747151,-1.189642,-0.171331,-0.411930,-0.501011,-0.325200,-0.948946,-0.694466,-0.847335,-1.034869
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101505,-0.909213,-0.932204,-0.499140,-0.819447,0.115991,1.274423,-1.706141,-0.227165,-0.388564,-0.375048,1.097832,-0.094594,-0.890648,-0.861310,-0.574047,0.098391,-0.462955,-0.156272,-0.246956,-0.723379,-0.679164,-0.135162,0.137949,-0.702629,-0.203359,-0.327298,0.096375,0.146825,0.966305
109089,0.261831,-0.098026,-0.075530,-0.550829,-0.822598,1.357534,-0.046525,-0.471492,0.398108,0.271047,-0.031145,-0.236100,-0.079350,-0.486966,-0.574047,0.770143,-0.443170,-0.038859,0.082596,-0.191343,0.303760,-0.072689,-0.411930,-0.375146,-0.325200,0.489829,-0.663225,-0.814795,0.966305
62950,0.380015,-0.426385,0.643037,0.071775,1.921522,1.191312,0.036456,1.971780,1.447004,0.055709,-0.347517,-0.271790,0.326319,-0.519968,-0.574047,-0.134734,-0.071179,-0.156272,0.017735,-0.469004,0.008079,-0.051317,0.867050,-0.744327,-0.325200,-0.225892,-0.209361,9.389552,-1.034869
18640,0.507575,1.521125,1.167749,0.507648,-0.740697,-0.930183,0.534341,-0.960146,-0.650788,1.249311,-0.357075,-0.271790,-0.445951,0.327509,-0.574047,-0.023206,0.010795,-0.156272,1.380053,0.396484,2.024814,-0.116530,0.885114,-0.263285,-0.325200,0.283450,-0.433309,-0.750518,-1.034869


In [13]:
MODEL_SPACES = {
    "log_reg": {
        "C": loguniform(1e-4, 5e2),
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
    },
    "knn": {
        "n_neighbors": randint(3, 121),
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    },
    "naive_bayes": {
        "var_smoothing": loguniform(1e-12, 1e-6),
    },
    "random_forest": {
        "max_depth": [None, 4, 6, 8, 12, 16, 20],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 15),
        "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
        "class_weight": [None, "balanced", "balanced_subsample"],
    },
    "xgboost": {
        "learning_rate": uniform(0.01, 0.29),
        "max_depth": randint(3, 10),
        "subsample": uniform(0.55, 0.45),
        "colsample_bytree": uniform(0.55, 0.45),
        "min_child_weight": randint(1, 12),
        "gamma": uniform(0.0, 3.0),
        "reg_lambda": loguniform(1e-3, 30),
    },
    "catboost": {
        "learning_rate": uniform(0.01, 0.29),
        "depth": randint(4, 11),
        "l2_leaf_reg": loguniform(1.0, 20.0),
        "random_strength": uniform(0.0, 2.0),
    },
    "adaboost": {
        "learning_rate": uniform(0.01, 1.4),
    },
    "lightgbm": {
        "learning_rate": uniform(0.01, 0.29),
        "num_leaves": randint(16, 140),
        "feature_fraction": uniform(0.55, 0.45),
        "bagging_fraction": uniform(0.55, 0.45),
        "min_child_samples": randint(5, 80),
        "reg_lambda": loguniform(1e-3, 30),
        "min_split_gain": uniform(0.0, 1.5),
    },
}


def model_is_available(name):
    if name == "xgboost":
        return xgb_available
    if name == "catboost":
        return cat_available
    if name == "lightgbm":
        return lgb_available
    return True


AVAILABLE_MODELS = [m for m in MODEL_SPACES if model_is_available(m)]
print("Models to train:", AVAILABLE_MODELS)

Models to train: ['log_reg', 'knn', 'naive_bayes', 'random_forest', 'xgboost', 'catboost', 'adaboost', 'lightgbm']


In [14]:
def safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        if p.ndim == 2:
            return np.clip(p[:, 1], 1e-6, 1 - 1e-6)
        return np.clip(p.reshape(-1), 1e-6, 1 - 1e-6)
    if hasattr(model, "decision_function"):
        return np.clip(expit(model.decision_function(X)), 1e-6, 1 - 1e-6)
    return np.clip(model.predict(X).astype(float), 1e-6, 1 - 1e-6)


def metric_pack(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.clip(np.asarray(y_prob), 1e-6, 1 - 1e-6)
    y_pred = (y_prob >= threshold).astype(int)

    if np.unique(y_true).size < 2:
        auc_val = 0.5
    else:
        auc_val = roc_auc_score(y_true, y_prob)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": auc_val,
        "logloss": log_loss(y_true, y_prob, labels=[0, 1]),
    }


def stage_score(metrics):
    auc = metrics.get("auc_mean", metrics.get("auc", 0.0))
    logloss = metrics.get("logloss_mean", metrics.get("logloss", 10.0))
    f1 = metrics.get("f1_mean", metrics.get("f1", 0.0))
    auc_std = metrics.get("auc_std", 0.0)
    logloss_std = metrics.get("logloss_std", 0.0)
    return (auc - 0.70 * logloss + 0.15 * f1) - (0.05 * auc_std + 0.02 * logloss_std)


def build_model(model_name, params, epoch_budget):
    p = deepcopy(params)

    if model_name == "log_reg":
        solver = p["solver"]
        penalty = p["penalty"]
        if penalty == "l1" and solver not in {"liblinear", "saga"}:
            solver = "liblinear"
        return LogisticRegression(
            C=float(p["C"]),
            penalty=penalty,
            solver=solver,
            class_weight=p.get("class_weight", "balanced"),
            max_iter=max(500, int(epoch_budget * 15)),
            random_state=RANDOM_SEED,
        )

    if model_name == "knn":
        return KNeighborsClassifier(
            n_neighbors=int(p["n_neighbors"]),
            weights=p["weights"],
            p=int(p["p"]),
        )

    if model_name == "naive_bayes":
        return GaussianNB(var_smoothing=float(p["var_smoothing"]))

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=int(epoch_budget),
            max_depth=p["max_depth"],
            min_samples_split=int(p["min_samples_split"]),
            min_samples_leaf=int(p["min_samples_leaf"]),
            max_features=p["max_features"],
            class_weight=p.get("class_weight", "balanced_subsample"),
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )

    if model_name == "xgboost":
        return XGBClassifier(
            n_estimators=int(epoch_budget),
            learning_rate=float(p["learning_rate"]),
            max_depth=int(p["max_depth"]),
            subsample=float(p["subsample"]),
            colsample_bytree=float(p["colsample_bytree"]),
            min_child_weight=float(p.get("min_child_weight", 1.0)),
            gamma=float(p.get("gamma", 0.0)),
            reg_lambda=float(p["reg_lambda"]),
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if model_name == "catboost":
        return CatBoostClassifier(
            iterations=int(epoch_budget),
            learning_rate=float(p["learning_rate"]),
            depth=int(p["depth"]),
            l2_leaf_reg=float(p["l2_leaf_reg"]),
            random_strength=float(p.get("random_strength", 1.0)),
            loss_function="Logloss",
            random_seed=RANDOM_SEED,
            verbose=False,
        )

    if model_name == "adaboost":
        return AdaBoostClassifier(
            n_estimators=int(epoch_budget),
            learning_rate=float(p["learning_rate"]),
            random_state=RANDOM_SEED,
        )

    if model_name == "lightgbm":
        return LGBMClassifier(
            n_estimators=int(epoch_budget),
            learning_rate=float(p["learning_rate"]),
            num_leaves=int(p["num_leaves"]),
            feature_fraction=float(p["feature_fraction"]),
            bagging_fraction=float(p["bagging_fraction"]),
            min_child_samples=int(p["min_child_samples"]),
            reg_lambda=float(p.get("reg_lambda", 0.0)),
            min_split_gain=float(p.get("min_split_gain", 0.0)),
            objective="binary",
            random_state=RANDOM_SEED,
            verbosity=-1,
            n_jobs=-1,
        )

    raise ValueError(f"Unsupported model: {model_name}")


def fit_model(model, Xtr, ytr, Xval=None, yval=None, early_stopping_rounds=None):
    model_name = type(model).__name__.lower()

    if "catboost" in model_name and Xval is not None:
        model.fit(
            Xtr,
            ytr,
            eval_set=(Xval, yval),
            early_stopping_rounds=early_stopping_rounds,
            verbose=False,
        )
        return model

    if "xgb" in model_name and Xval is not None:
        model.fit(
            Xtr,
            ytr,
            eval_set=[(Xval, yval)],
            verbose=False,
        )
        return model

    if "lgbm" in model_name and Xval is not None and lgb is not None:
        callbacks = []
        if early_stopping_rounds is not None:
            callbacks = [lgb.early_stopping(early_stopping_rounds, verbose=False)]
        model.fit(
            Xtr,
            ytr,
            eval_set=[(Xval, yval)],
            eval_metric="binary_logloss",
            callbacks=callbacks,
        )
        return model

    model.fit(Xtr, ytr)
    return model


def evaluate_params_cv(model_name, params, X_data, y_data, epoch_budget, n_splits=5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    y_array = np.asarray(y_data).astype(int)
    rows = []

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X_data, y_array), start=1):
        Xtr = X_data.iloc[tr_idx]
        Xva = X_data.iloc[va_idx]
        ytr = y_data.iloc[tr_idx]
        yva = y_data.iloc[va_idx]

        model = build_model(model_name, params, epoch_budget=epoch_budget)
        model = fit_model(
            model,
            Xtr,
            ytr,
            Xva,
            yva,
            early_stopping_rounds=10 if epoch_budget >= STAGE2_EPOCHS else None,
        )
        p_val = safe_predict_proba(model, Xva)
        met = metric_pack(yva, p_val, threshold=0.5)
        met["fold"] = fold
        rows.append(met)

    fold_df = pd.DataFrame(rows)
    summary = {
        "auc_mean": float(fold_df["auc"].mean()),
        "auc_std": float(fold_df["auc"].std(ddof=0)),
        "logloss_mean": float(fold_df["logloss"].mean()),
        "logloss_std": float(fold_df["logloss"].std(ddof=0)),
        "f1_mean": float(fold_df["f1"].mean()),
        "f1_std": float(fold_df["f1"].std(ddof=0)),
    }
    summary["stage_score"] = stage_score(summary)
    return summary


def sample_stage1_params(model_name, n_trials):
    return list(ParameterSampler(MODEL_SPACES[model_name], n_iter=n_trials, random_state=RANDOM_SEED))


def dedupe_param_dicts(params_list):
    uniq = []
    seen = set()
    for p in params_list:
        if not isinstance(p, dict):
            continue
        key = json.dumps(p, sort_keys=True)
        if key not in seen:
            seen.add(key)
            uniq.append(p)
    return uniq

In [15]:
stage1_results = {}
stage1_top_params = {}

for model_name in AVAILABLE_MODELS:
    trials = sample_stage1_params(model_name, STAGE1_TRIALS_PER_MODEL)
    rows = []
    print(
        f"Stage 1 CV search -> {model_name} "
        f"(epochs={STAGE1_EPOCHS}, folds={CV_FOLDS_STAGE1}, trials={len(trials)})"
    )

    for i, params in enumerate(trials, start=1):
        try:
            cv_met = evaluate_params_cv(
                model_name,
                params,
                X_train_final,
                y_train,
                epoch_budget=STAGE1_EPOCHS,
                n_splits=CV_FOLDS_STAGE1,
            )
            rows.append(
                {
                    "trial": i,
                    "params": params,
                    "auc_mean": cv_met["auc_mean"],
                    "auc_std": cv_met["auc_std"],
                    "logloss_mean": cv_met["logloss_mean"],
                    "logloss_std": cv_met["logloss_std"],
                    "f1_mean": cv_met["f1_mean"],
                    "f1_std": cv_met["f1_std"],
                    "stage_score": cv_met["stage_score"],
                }
            )
        except Exception as e:
            rows.append({"trial": i, "params": params, "error": str(e), "stage_score": -999.0})

        if i % 10 == 0 or i == len(trials):
            print(f"  completed {i}/{len(trials)}")

    df_stage = pd.DataFrame(rows).sort_values(["stage_score", "auc_mean"], ascending=False).reset_index(drop=True)
    stage1_results[model_name] = df_stage
    stage1_top_params[model_name] = df_stage.head(TOP_K_STAGE1)["params"].tolist()

stage1_summary = pd.DataFrame(
    [
        {
            "model": m,
            "best_stage1_auc_mean": stage1_results[m].loc[0, "auc_mean"] if "auc_mean" in stage1_results[m].columns else np.nan,
            "best_stage1_logloss_mean": stage1_results[m].loc[0, "logloss_mean"] if "logloss_mean" in stage1_results[m].columns else np.nan,
            "best_stage1_f1_mean": stage1_results[m].loc[0, "f1_mean"] if "f1_mean" in stage1_results[m].columns else np.nan,
            "trials": len(stage1_results[m]),
        }
        for m in AVAILABLE_MODELS
    ]
).sort_values("best_stage1_auc_mean", ascending=False)

stage1_summary

Stage 1 CV search -> log_reg (epochs=40, folds=5, trials=140)
  completed 10/140
  completed 20/140
  completed 30/140
  completed 40/140
  completed 50/140
  completed 60/140
  completed 70/140
  completed 80/140
  completed 90/140
  completed 100/140
  completed 110/140
  completed 120/140
  completed 130/140
  completed 140/140
Stage 1 CV search -> knn (epochs=40, folds=5, trials=140)
  completed 10/140
  completed 20/140
  completed 30/140
  completed 40/140
  completed 50/140
  completed 60/140
  completed 70/140
  completed 80/140
  completed 90/140
  completed 100/140
  completed 110/140
  completed 120/140
  completed 130/140
  completed 140/140
Stage 1 CV search -> naive_bayes (epochs=40, folds=5, trials=140)
  completed 10/140
  completed 20/140
  completed 30/140
  completed 40/140
  completed 50/140
  completed 60/140
  completed 70/140
  completed 80/140
  completed 90/140
  completed 100/140
  completed 110/140
  completed 120/140
  completed 130/140
  completed 140/140
S

,model,best_stage1_auc_mean,best_stage1_logloss_mean,best_stage1_f1_mean,trials
5,catboost,0.826890,0.466681,0.659811,140
7,lightgbm,0.826668,0.466553,0.660169,140
4,xgboost,0.826117,0.466863,0.662306,140
3,random_forest,0.826050,0.467044,0.660875,140
0,log_reg,0.822425,0.487766,0.598691,140
6,adaboost,0.819028,0.502489,0.658058,140
1,knn,0.809723,0.489947,0.597835,140
2,naive_bayes,0.776389,0.790712,0.616545,140


In [16]:
def refine_candidates(base_params_list, n_refinements_per_base=4):
    refined = []
    for base in base_params_list:
        if not isinstance(base, dict):
            continue

        refined.append(deepcopy(base))
        for _ in range(n_refinements_per_base):
            cand = {}
            for k, v in base.items():
                if isinstance(v, (int, np.integer)):
                    factor = np.random.uniform(0.65, 1.35)
                    cand[k] = max(1, int(round(v * factor)))
                elif isinstance(v, (float, np.floating)):
                    factor = np.random.uniform(0.65, 1.35)
                    cand[k] = max(1e-6, float(v * factor))
                else:
                    cand[k] = v
            refined.append(cand)
    return dedupe_param_dicts(refined)


stage2_results = {}
best_configs = {}

for model_name in AVAILABLE_MODELS:
    local_refined = refine_candidates(
        stage1_top_params.get(model_name, []),
        n_refinements_per_base=STAGE2_REFINEMENTS_PER_TOP_CONFIG,
    )
    exploration_trials = sample_stage1_params(
        model_name,
        max(16, STAGE1_TRIALS_PER_MODEL // 3),
    )
    candidates = dedupe_param_dicts(local_refined + exploration_trials)

    rows = []
    print(
        f"Stage 2 CV search -> {model_name} "
        f"(epochs={STAGE2_EPOCHS}, folds={CV_FOLDS_STAGE2}, candidates={len(candidates)})"
    )

    for i, params in enumerate(candidates, start=1):
        try:
            cv_met = evaluate_params_cv(
                model_name,
                params,
                X_train_final,
                y_train,
                epoch_budget=STAGE2_EPOCHS,
                n_splits=CV_FOLDS_STAGE2,
            )
            rows.append(
                {
                    "trial": i,
                    "params": params,
                    "auc_mean": cv_met["auc_mean"],
                    "auc_std": cv_met["auc_std"],
                    "logloss_mean": cv_met["logloss_mean"],
                    "logloss_std": cv_met["logloss_std"],
                    "f1_mean": cv_met["f1_mean"],
                    "f1_std": cv_met["f1_std"],
                    "stage_score": cv_met["stage_score"],
                }
            )
        except Exception as e:
            rows.append({"trial": i, "params": params, "error": str(e), "stage_score": -999.0})

        if i % 10 == 0 or i == len(candidates):
            print(f"  completed {i}/{len(candidates)}")

    df_stage2 = pd.DataFrame(rows).sort_values(["stage_score", "auc_mean"], ascending=False).reset_index(drop=True)
    stage2_results[model_name] = df_stage2
    best_configs[model_name] = df_stage2.head(TOP_K_STAGE2)["params"].tolist()

stage2_summary = pd.DataFrame(
    [
        {
            "model": m,
            "best_stage2_auc_mean": stage2_results[m].loc[0, "auc_mean"] if "auc_mean" in stage2_results[m].columns else np.nan,
            "best_stage2_logloss_mean": stage2_results[m].loc[0, "logloss_mean"] if "logloss_mean" in stage2_results[m].columns else np.nan,
            "best_stage2_f1_mean": stage2_results[m].loc[0, "f1_mean"] if "f1_mean" in stage2_results[m].columns else np.nan,
            "candidates": len(stage2_results[m]),
        }
        for m in AVAILABLE_MODELS
    ]
).sort_values("best_stage2_auc_mean", ascending=False)

stage2_summary

Stage 2 CV search -> log_reg (epochs=120, folds=6, candidates=330)
  completed 10/330
  completed 20/330
  completed 30/330
  completed 40/330
  completed 50/330
  completed 60/330
  completed 70/330
  completed 80/330
  completed 90/330
  completed 100/330
  completed 110/330
  completed 120/330
  completed 130/330
  completed 140/330
  completed 150/330
  completed 160/330
  completed 170/330
  completed 180/330
  completed 190/330
  completed 200/330
  completed 210/330
  completed 220/330
  completed 230/330
  completed 240/330
  completed 250/330
  completed 260/330
  completed 270/330
  completed 280/330
  completed 290/330
  completed 300/330
  completed 310/330
  completed 320/330
  completed 330/330
Stage 2 CV search -> knn (epochs=120, folds=6, candidates=129)
  completed 10/129
  completed 20/129
  completed 30/129
  completed 40/129
  completed 50/129
  completed 60/129
  completed 70/129
  completed 80/129
  completed 90/129
  completed 100/129
  completed 110/129
  comple

,model,best_stage2_auc_mean,best_stage2_logloss_mean,best_stage2_f1_mean,candidates
5,catboost,0.827297,0.466039,0.657796,333
7,lightgbm,0.826976,0.466107,0.659727,333
4,xgboost,0.826780,0.466235,0.664413,331
3,random_forest,0.826152,0.466702,0.663150,333
0,log_reg,0.822417,0.487774,0.598762,330
6,adaboost,0.820963,0.505527,0.648647,332
1,knn,0.810268,0.488676,0.595435,129
2,naive_bayes,0.777248,0.784329,0.616886,56


In [17]:
final_models = {}
selected_final_params = {}
final_rows = []
final_selection_rows = []

for model_name in AVAILABLE_MODELS:
    top_list = best_configs.get(model_name, [])
    if not top_list:
        continue

    print(f"Final selection -> {model_name} (candidates={len(top_list)}, epochs={FINAL_EPOCHS})")

    best_model_obj = None
    best_params = None
    best_valid_score = -np.inf
    best_valid_met = None
    best_valid_probs = None

    for i, params in enumerate(top_list, start=1):
        try:
            model = build_model(model_name, params, epoch_budget=FINAL_EPOCHS)
            model = fit_model(
                model,
                X_train_final,
                y_train,
                X_valid_final,
                y_valid,
                early_stopping_rounds=15,
            )

            p_valid = safe_predict_proba(model, X_valid_final)
            valid_met = metric_pack(y_valid, p_valid, threshold=0.5)
            valid_score = stage_score(valid_met)

            final_selection_rows.append(
                {
                    "model": model_name,
                    "candidate_rank": i,
                    "params": params,
                    "valid_auc": valid_met["auc"],
                    "valid_logloss": valid_met["logloss"],
                    "valid_f1": valid_met["f1"],
                    "valid_stage_score": valid_score,
                }
            )

            if valid_score > best_valid_score:
                best_valid_score = valid_score
                best_model_obj = model
                best_params = params
                best_valid_met = valid_met
                best_valid_probs = p_valid
        except Exception as e:
            final_selection_rows.append(
                {
                    "model": model_name,
                    "candidate_rank": i,
                    "params": params,
                    "error": str(e),
                    "valid_stage_score": -999.0,
                }
            )

    if best_model_obj is None:
        print(f"Final train skipped for {model_name}: all candidates failed")
        continue

    final_models[model_name] = best_model_obj
    selected_final_params[model_name] = best_params
    joblib.dump(best_model_obj, MODEL_DIR / f"{model_name}.joblib")

    p_test = safe_predict_proba(best_model_obj, X_test_final)
    test_met = metric_pack(y_test, p_test, threshold=0.5)

    final_rows.append(
        {
            "model": model_name,
            "valid_auc": best_valid_met["auc"],
            "valid_logloss": best_valid_met["logloss"],
            "valid_f1": best_valid_met["f1"],
            "valid_stage_score": best_valid_score,
            "test_auc": test_met["auc"],
            "test_logloss": test_met["logloss"],
            "test_f1": test_met["f1"],
            "params": best_params,
        }
    )

final_results_df = pd.DataFrame(final_rows).sort_values(["valid_stage_score", "valid_auc"], ascending=False).reset_index(drop=True)
final_selection_df = pd.DataFrame(final_selection_rows).sort_values(["model", "valid_stage_score"], ascending=[True, False]).reset_index(drop=True)
final_selection_path = ARTIFACT_DIR / "final_candidate_selection_2015_rebuild.csv"
final_selection_df.to_csv(final_selection_path, index=False)

print(f"Saved final candidate comparisons: {final_selection_path}")
final_results_df

Final selection -> log_reg (candidates=4, epochs=500)
Final selection -> knn (candidates=4, epochs=500)
Final selection -> naive_bayes (candidates=4, epochs=500)
Final selection -> random_forest (candidates=4, epochs=500)
Final selection -> xgboost (candidates=4, epochs=500)
Final selection -> catboost (candidates=4, epochs=500)
Final selection -> adaboost (candidates=4, epochs=500)
Final selection -> lightgbm (candidates=4, epochs=500)
Saved final candidate comparisons: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\final_candidate_selection_2015_rebuild.csv


,model,valid_auc,valid_logloss,valid_f1,valid_stage_score,test_auc,test_logloss,test_f1,params
0,random_forest,0.829160,0.462797,0.673115,0.606170,0.817859,0.476522,0.657701,"{'class_weight': None, 'max_depth': 5, 'max_fe..."
1,lightgbm,0.829756,0.462683,0.667853,0.606056,0.818334,0.475545,0.652509,"{'bagging_fraction': 0.553264005645652, 'featu..."
2,catboost,0.829767,0.462784,0.667448,0.605935,0.818435,0.475645,0.649120,"{'depth': 5, 'l2_leaf_reg': 16.848715461407302..."
3,xgboost,0.828716,0.463746,0.668937,0.604434,0.817847,0.476447,0.652562,"{'colsample_bytree': 0.991975754498246, 'gamma..."
4,log_reg,0.824365,0.485528,0.606145,0.575417,0.814609,0.495519,0.592762,"{'C': 0.0070439244589380005, 'class_weight': N..."
5,adaboost,0.825902,0.517290,0.652127,0.561618,0.815422,0.522430,0.637970,{'learning_rate': 0.1467409596089374}
6,knn,0.810878,0.487378,0.604009,0.560315,0.804310,0.493526,0.596379,"{'n_neighbors': 140, 'p': 1, 'weights': 'unifo..."
7,naive_bayes,0.777157,0.779857,0.622008,0.324558,0.775408,0.783755,0.615078,{'var_smoothing': 1.1244779162047178e-06}


In [18]:
def expected_calibration_error(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true)
    y_prob = np.clip(np.asarray(y_prob), 1e-6, 1 - 1e-6)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.digitize(y_prob, bins) - 1

    ece = 0.0
    for i in range(n_bins):
        mask = bin_idx == i
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def platt_calibration(p_fit, y_fit, p_eval):
    lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    lr.fit(p_fit.reshape(-1, 1), y_fit)
    return np.clip(lr.predict_proba(p_eval.reshape(-1, 1))[:, 1], 1e-6, 1 - 1e-6)


def isotonic_calibration(p_fit, y_fit, p_eval):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_fit, y_fit)
    return np.clip(iso.predict(p_eval), 1e-6, 1 - 1e-6)


def venn_abers_calibration(p_fit, y_fit, p_eval):
    if not venn_available:
        raise RuntimeError("venn-abers is not installed.")
    va = VennAbers()
    va.fit(np.column_stack([1.0 - p_fit, p_fit]), np.asarray(y_fit))
    _, p1 = va.predict_proba(np.column_stack([1.0 - p_eval, p_eval]))
    p1 = np.asarray(p1)
    if p1.ndim == 2:
        return np.clip(p1[:, 1], 1e-6, 1 - 1e-6)
    return np.clip(p1.reshape(-1), 1e-6, 1 - 1e-6)


def apply_calibration(method, p_fit, y_fit, p_eval):
    if method == "base":
        return np.clip(np.asarray(p_eval), 1e-6, 1 - 1e-6)
    if method == "platt":
        return platt_calibration(p_fit, y_fit, p_eval)
    if method == "isotonic":
        return isotonic_calibration(p_fit, y_fit, p_eval)
    if method == "venn_abers":
        return venn_abers_calibration(p_fit, y_fit, p_eval)
    raise ValueError(f"Unknown calibration method: {method}")


calibration_rows = []
calibrated_test_probs = {}

for model_name, model in final_models.items():
    p_valid_base = safe_predict_proba(model, X_valid_final)
    p_test_base = safe_predict_proba(model, X_test_final)

    methods = ["base", "platt", "isotonic"]
    if venn_available:
        methods.append("venn_abers")

    p_fit, p_eval, y_fit, y_eval = train_test_split(
        p_valid_base,
        y_valid.values,
        test_size=0.50,
        random_state=RANDOM_SEED,
        stratify=y_valid.values,
    )

    for method in methods:
        try:
            # Selection metrics are computed on validation holdout only.
            p_eval_cal = apply_calibration(method, p_fit, y_fit, p_eval)
            eval_m = metric_pack(y_eval, p_eval_cal, threshold=0.5)

            # Final test probabilities are generated after refitting on full validation split.
            p_test_cal = apply_calibration(method, p_valid_base, y_valid.values, p_test_base)

            calibration_rows.append(
                {
                    "model": model_name,
                    "method": method,
                    "cal_logloss": eval_m["logloss"],
                    "cal_ece": expected_calibration_error(y_eval, p_eval_cal),
                    "cal_auc": eval_m["auc"],
                    "cal_f1": eval_m["f1"],
                }
            )
            calibrated_test_probs[(model_name, method)] = p_test_cal
        except Exception as e:
            print(f"{model_name}: {method} skipped ({e})")

calibration_df = pd.DataFrame(calibration_rows)
calibration_df["logloss_rank"] = calibration_df["cal_logloss"].rank(method="min")
calibration_df["ece_rank"] = calibration_df["cal_ece"].rank(method="min")
calibration_df["avg_rank"] = (calibration_df["logloss_rank"] + calibration_df["ece_rank"]) / 2.0
calibration_df = calibration_df.sort_values(["avg_rank", "cal_logloss", "cal_ece"]).reset_index(drop=True)

calibration_path = ARTIFACT_DIR / "calibration_results_2015_rebuild.csv"
calibration_df.to_csv(calibration_path, index=False)

calibration_df.head(20)

,model,method,cal_logloss,cal_ece,cal_auc,cal_f1,logloss_rank,ece_rank,avg_rank
0,random_forest,base,0.463776,0.011807,0.827758,0.669881,1.0,6.0,3.5
1,catboost,base,0.464291,0.010299,0.827569,0.664536,3.0,4.0,3.5
2,lightgbm,base,0.463811,0.012367,0.827712,0.665852,2.0,9.0,5.5
3,xgboost,base,0.465304,0.011838,0.826978,0.665209,5.0,7.0,6.0
4,catboost,isotonic,0.465480,0.012277,0.827238,0.667637,8.0,8.0,8.0
5,xgboost,isotonic,0.467260,0.012612,0.826014,0.665294,12.0,10.0,11.0
6,lightgbm,venn_abers,0.465109,0.015813,0.827077,0.674083,4.0,20.0,12.0
7,random_forest,isotonic,0.465441,0.014454,0.827292,0.672777,7.0,17.0,12.0
8,catboost,venn_abers,0.466262,0.013480,0.827320,0.667556,10.0,14.0,12.0
9,xgboost,venn_abers,0.467048,0.013143,0.826025,0.665053,11.0,13.0,12.0


In [19]:
best_cal = calibration_df.iloc[0]
best_key = (best_cal["model"], best_cal["method"])
best_test_probs = calibrated_test_probs[best_key]
best_model_name = best_cal["model"]
best_method = best_cal["method"]
best_model = final_models[best_model_name]

best_metrics = metric_pack(y_test, best_test_probs, threshold=0.5)
best_summary = {
    "best_model": best_model_name,
    "calibration_method": best_method,
    "selection_cal_logloss": float(best_cal["cal_logloss"]),
    "selection_cal_ece": float(best_cal["cal_ece"]),
    "test_accuracy": best_metrics["accuracy"],
    "test_precision": best_metrics["precision"],
    "test_recall": best_metrics["recall"],
    "test_f1": best_metrics["f1"],
    "test_auc": best_metrics["auc"],
    "test_logloss": best_metrics["logloss"],
    "test_ece": expected_calibration_error(y_test.values, best_test_probs),
}

summary_path = ARTIFACT_DIR / "best_calibrated_summary_2015_rebuild.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(best_summary, f, indent=2)

print("Best calibrated configuration (selected on validation holdout):")
print(pd.DataFrame([best_summary]).to_string(index=False))
print(f"Saved: {calibration_path}")
print(f"Saved: {summary_path}")

Best calibrated configuration (selected on validation holdout):
   best_model calibration_method  selection_cal_logloss  selection_cal_ece  test_accuracy  test_precision  test_recall  test_f1  test_auc  test_logloss  test_ece
random_forest               base               0.463776           0.011807       0.749724         0.60873     0.715241 0.657701  0.817859      0.476522   0.00708
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\calibration_results_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\best_calibrated_summary_2015_rebuild.json


In [20]:
stack_base_pool = final_results_df.sort_values(["valid_stage_score", "valid_auc"], ascending=False)["model"].tolist()
stack_members = stack_base_pool[: min(STACK_TOP_BASE_MODELS, len(stack_base_pool))]
if len(stack_members) < 2:
    raise RuntimeError("Need at least 2 trained models for stacking optimization.")

print("Stacking base models:", stack_members)


def build_stack_feature_matrices(member_names, n_splits=5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    y_train_arr = np.asarray(y_train).astype(int)

    P_train_oof = np.zeros((len(X_train_final), len(member_names)))
    P_valid_meta = np.zeros((len(X_valid_final), len(member_names)))
    P_test_meta = np.zeros((len(X_test_final), len(member_names)))

    for col_idx, model_name in enumerate(member_names):
        if model_name not in selected_final_params:
            raise RuntimeError(f"Missing selected params for stack member: {model_name}")

        params = selected_final_params[model_name]
        valid_fold_preds = []
        test_fold_preds = []
        print(f"  OOF stack features -> {model_name}")

        for _, (tr_idx, va_idx) in enumerate(splitter.split(X_train_final, y_train_arr), start=1):
            Xtr = X_train_final.iloc[tr_idx]
            Xva = X_train_final.iloc[va_idx]
            ytr = y_train.iloc[tr_idx]
            yva = y_train.iloc[va_idx]

            model = build_model(model_name, params, epoch_budget=FINAL_EPOCHS)
            model = fit_model(
                model,
                Xtr,
                ytr,
                Xva,
                yva,
                early_stopping_rounds=15,
            )

            P_train_oof[va_idx, col_idx] = safe_predict_proba(model, Xva)
            valid_fold_preds.append(safe_predict_proba(model, X_valid_final))
            test_fold_preds.append(safe_predict_proba(model, X_test_final))

        P_valid_meta[:, col_idx] = np.mean(np.column_stack(valid_fold_preds), axis=1)
        P_test_meta[:, col_idx] = np.mean(np.column_stack(test_fold_preds), axis=1)

    return P_train_oof, P_valid_meta, P_test_meta


P_train_oof, P_valid_meta, P_test_meta = build_stack_feature_matrices(
    stack_members,
    n_splits=STACK_OOF_FOLDS,
)

meta_space = {
    "C": loguniform(1e-4, 1e3),
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "class_weight": [None, "balanced"],
}
meta_trials = list(ParameterSampler(meta_space, n_iter=STACK_META_TRIALS, random_state=RANDOM_SEED))

meta_rows = []
best_meta_model = None
best_meta_params = None
best_meta_valid_probs = None
best_meta_test_probs = None
best_meta_score = -np.inf

for i, params in enumerate(meta_trials, start=1):
    try:
        penalty = params["penalty"]
        solver = params["solver"]
        if penalty == "l1" and solver not in {"liblinear", "saga"}:
            solver = "liblinear"

        meta_model = LogisticRegression(
            C=float(params["C"]),
            penalty=penalty,
            solver=solver,
            class_weight=params.get("class_weight", None),
            max_iter=8000,
            random_state=RANDOM_SEED,
        )
        meta_model.fit(P_train_oof, y_train.values)

        p_valid = np.clip(meta_model.predict_proba(P_valid_meta)[:, 1], 1e-6, 1 - 1e-6)
        valid_met = metric_pack(y_valid, p_valid, threshold=0.5)
        valid_score = stage_score(valid_met)

        meta_rows.append(
            {
                "strategy": "meta_logistic",
                "trial": i,
                "params": params,
                "valid_auc": valid_met["auc"],
                "valid_logloss": valid_met["logloss"],
                "valid_f1": valid_met["f1"],
                "valid_stage_score": valid_score,
            }
        )

        if valid_score > best_meta_score:
            best_meta_score = valid_score
            best_meta_model = meta_model
            best_meta_params = params
            best_meta_valid_probs = p_valid
            best_meta_test_probs = np.clip(meta_model.predict_proba(P_test_meta)[:, 1], 1e-6, 1 - 1e-6)
    except Exception as e:
        meta_rows.append({"strategy": "meta_logistic", "trial": i, "error": str(e), "valid_stage_score": -999.0})

    if i % 25 == 0 or i == len(meta_trials):
        print(f"  meta search progress: {i}/{len(meta_trials)}")

rng = np.random.default_rng(RANDOM_SEED)
blend_rows = []
best_blend_weights = None
best_blend_valid_probs = None
best_blend_test_probs = None
best_blend_score = -np.inf

for i in range(STACK_BLEND_TRIALS):
    weights = rng.dirichlet(np.ones(len(stack_members)))
    p_valid = np.clip(P_valid_meta @ weights, 1e-6, 1 - 1e-6)
    valid_met = metric_pack(y_valid, p_valid, threshold=0.5)
    valid_score = stage_score(valid_met)

    blend_rows.append(
        {
            "strategy": "weighted_blend",
            "trial": i + 1,
            "weights_json": json.dumps(weights.tolist()),
            "valid_auc": valid_met["auc"],
            "valid_logloss": valid_met["logloss"],
            "valid_f1": valid_met["f1"],
            "valid_stage_score": valid_score,
        }
    )

    if valid_score > best_blend_score:
        best_blend_score = valid_score
        best_blend_weights = weights
        best_blend_valid_probs = p_valid
        best_blend_test_probs = np.clip(P_test_meta @ weights, 1e-6, 1 - 1e-6)

    if (i + 1) % 500 == 0 or (i + 1) == STACK_BLEND_TRIALS:
        print(f"  blend search progress: {i + 1}/{STACK_BLEND_TRIALS}")

if best_meta_valid_probs is None and best_blend_valid_probs is None:
    raise RuntimeError("Stack optimization failed: no valid strategy found.")

if best_meta_valid_probs is not None and (best_meta_score >= best_blend_score):
    stack_strategy = "meta_logistic"
    stack_strategy_details = best_meta_params
    p_stack_valid = best_meta_valid_probs
    p_stack_test = best_meta_test_probs
else:
    stack_strategy = "weighted_blend"
    stack_strategy_details = {"weights": best_blend_weights.tolist()}
    p_stack_valid = best_blend_valid_probs
    p_stack_test = best_blend_test_probs

print(f"Best raw stack strategy selected on validation: {stack_strategy}")

stack_search_df = pd.concat([pd.DataFrame(meta_rows), pd.DataFrame(blend_rows)], ignore_index=True)
stack_search_df = stack_search_df.sort_values("valid_stage_score", ascending=False).reset_index(drop=True)
stack_search_path = ARTIFACT_DIR / "stack_search_results_2015_rebuild.csv"
stack_search_df.to_csv(stack_search_path, index=False)

methods = ["base", "platt", "isotonic"]
if venn_available:
    methods.append("venn_abers")

p_fit, p_eval, y_fit, y_eval = train_test_split(
    p_stack_valid,
    y_valid.values,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_valid.values,
 )

stack_cal_rows = []
stack_test_probs = {}

for method in methods:
    try:
        p_eval_cal = apply_calibration(method, p_fit, y_fit, p_eval)
        eval_m = metric_pack(y_eval, p_eval_cal, threshold=0.5)

        p_test_cal = apply_calibration(method, p_stack_valid, y_valid.values, p_stack_test)
        stack_test_probs[method] = p_test_cal

        stack_cal_rows.append(
            {
                "model": "stack_meta",
                "strategy": stack_strategy,
                "method": method,
                "cal_logloss": eval_m["logloss"],
                "cal_ece": expected_calibration_error(y_eval, p_eval_cal),
                "cal_auc": eval_m["auc"],
                "cal_f1": eval_m["f1"],
            }
        )
    except Exception as e:
        print(f"stack_meta {method} skipped: {e}")

stack_cal_df = pd.DataFrame(stack_cal_rows)
stack_cal_df["logloss_rank"] = stack_cal_df["cal_logloss"].rank(method="min")
stack_cal_df["ece_rank"] = stack_cal_df["cal_ece"].rank(method="min")
stack_cal_df["avg_rank"] = (stack_cal_df["logloss_rank"] + stack_cal_df["ece_rank"]) / 2.0
stack_cal_df = stack_cal_df.sort_values(["avg_rank", "cal_logloss", "cal_ece"]).reset_index(drop=True)

best_stack_method = stack_cal_df.iloc[0]["method"]
best_stack_test_probs = stack_test_probs[best_stack_method]
best_stack_metrics = metric_pack(y_test, best_stack_test_probs, threshold=0.5)
best_stack_metrics["test_ece"] = expected_calibration_error(y_test.values, best_stack_test_probs)

print("Best stack calibration method (selected on validation holdout):", best_stack_method)
print("Best stack test metrics:")
print(pd.DataFrame([best_stack_metrics]).to_string(index=False))

stack_cal_path = ARTIFACT_DIR / "stack_calibration_results_2015_rebuild.csv"
stack_cal_df.to_csv(stack_cal_path, index=False)
print(f"Saved stack search results: {stack_search_path}")
stack_cal_df

Stacking base models: ['random_forest', 'lightgbm', 'catboost', 'xgboost', 'log_reg', 'adaboost', 'knn']
  OOF stack features -> random_forest
  OOF stack features -> lightgbm
  OOF stack features -> catboost
  OOF stack features -> xgboost
  OOF stack features -> log_reg
  OOF stack features -> adaboost
  OOF stack features -> knn
  meta search progress: 25/400
  meta search progress: 50/400
  meta search progress: 75/400
  meta search progress: 100/400
  meta search progress: 125/400
  meta search progress: 150/400
  meta search progress: 175/400
  meta search progress: 200/400
  meta search progress: 225/400
  meta search progress: 250/400
  meta search progress: 275/400
  meta search progress: 300/400
  meta search progress: 325/400
  meta search progress: 350/400
  meta search progress: 375/400
  meta search progress: 400/400
  blend search progress: 500/7000
  blend search progress: 1000/7000
  blend search progress: 1500/7000
  blend search progress: 2000/7000
  blend search pro

,model,strategy,method,cal_logloss,cal_ece,cal_auc,cal_f1,logloss_rank,ece_rank,avg_rank
0,stack_meta,weighted_blend,base,0.463713,0.013823,0.828297,0.665526,1.0,2.0,1.5
1,stack_meta,weighted_blend,isotonic,0.468517,0.013813,0.827504,0.671230,3.0,1.0,2.0
2,stack_meta,weighted_blend,venn_abers,0.466802,0.014611,0.827617,0.670825,2.0,3.0,2.5
3,stack_meta,weighted_blend,platt,0.469765,0.031962,0.828297,0.656817,4.0,4.0,4.0


In [21]:
if not shap_available:
    print("SHAP skipped: package not installed.")
else:
    def is_tree_model(model):
        name = type(model).__name__.lower()
        return any(tok in name for tok in ["xgb", "lgb", "catboost", "forest", "boost", "tree"])

    explain_model_name = best_model_name if is_tree_model(best_model) else None
    if explain_model_name is None:
        for k, m in final_models.items():
            if is_tree_model(m):
                explain_model_name = k
                break

    if explain_model_name is None:
        print("SHAP skipped: no tree-based model available.")
    else:
        explain_model = final_models[explain_model_name]
        X_bg = X_train_final.sample(min(2000, len(X_train_final)), random_state=RANDOM_SEED)

        try:
            explainer = shap.TreeExplainer(explain_model)
            shap_values = explainer.shap_values(X_bg)
            if isinstance(shap_values, list):
                sv = shap_values[1] if len(shap_values) > 1 else shap_values[0]
            else:
                sv = shap_values

            shap_df = pd.DataFrame(
                {
                    "feature": X_bg.columns,
                    "mean_abs_shap": np.abs(np.asarray(sv)).mean(axis=0),
                }
            ).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

            shap_path = EXPLAIN_DIR / "shap_importance_2015_rebuild.csv"
            shap_df.to_csv(shap_path, index=False)

            print(f"SHAP model used: {explain_model_name}")
            print(shap_df.head(20).to_string(index=False))
            print(f"Saved: {shap_path}")
        except Exception as e:
            print(f"SHAP skipped due to runtime issue: {e}")

SHAP skipped due to runtime issue: Per-column arrays must each be 1-dimensional


In [22]:
if not lime_available:
    print("LIME skipped: package not installed.")
else:
    lime_model = best_model
    lime_model_name = best_model_name

    lime_explainer = LimeTabularExplainer(
        training_data=X_train_final.values,
        feature_names=X_train_final.columns.tolist(),
        class_names=["class_0", "class_1"],
        mode="classification",
        discretize_continuous=True,
        random_state=RANDOM_SEED,
    )

    lime_rows = []
    explain_cases = min(5, len(X_test_final))

    for i in range(explain_cases):
        instance = X_test_final.iloc[i].values
        exp = lime_explainer.explain_instance(
            instance,
            lime_model.predict_proba,
            num_features=min(20, X_train_final.shape[1]),
        )

        html_path = LIME_DIR / f"lime_case_{i + 1}.html"
        exp.save_to_file(str(html_path))

        for rule, weight in exp.as_list():
            lime_rows.append(
                {
                    "case_number": i + 1,
                    "model": lime_model_name,
                    "feature_rule": rule,
                    "weight": float(weight),
                }
            )

    lime_local_df = pd.DataFrame(lime_rows)
    lime_local_path = EXPLAIN_DIR / "lime_local_weights_2015_rebuild.csv"
    lime_local_df.to_csv(lime_local_path, index=False)

    lime_global_df = (
        lime_local_df.assign(abs_weight=lambda d: d["weight"].abs())
        .groupby("feature_rule", as_index=False)["abs_weight"]
        .mean()
        .rename(columns={"abs_weight": "mean_abs_weight"})
        .sort_values("mean_abs_weight", ascending=False)
        .reset_index(drop=True)
    )
    lime_global_path = EXPLAIN_DIR / "lime_global_importance_2015_rebuild.csv"
    lime_global_df.to_csv(lime_global_path, index=False)

    print("LIME global feature rules:")
    print(lime_global_df.head(20).to_string(index=False))
    print(f"Saved: {lime_local_path}")
    print(f"Saved: {lime_global_path}")
    print(f"Saved HTML explanations in: {LIME_DIR}")

LIME global feature rules:
                    feature_rule  mean_abs_weight
               num__age <= -0.88         0.356222
       -0.88 < num__age <= -0.23         0.222732
        -0.23 < num__age <= 0.79         0.141558
              num__fg26 <= -0.57         0.015241
      -0.57 < num__fg26 <= -0.21         0.014779
                 num__fg7 > 0.22         0.013771
                 num__fg6 > 0.27         0.012306
              num__fg27 <= -0.17         0.011991
      -0.17 < num__fg27 <= -0.05         0.011628
-0.47 < num__Total_Iron <= -0.15         0.011560
               num__fg4 <= -0.60         0.011508
        -0.63 < num__cu <= -0.07         0.010504
       -0.32 < num__fg5 <= -0.06         0.010317
         -0.07 < num__cu <= 0.49         0.010169
          num__Total_Iron > 0.24         0.009510
        -0.22 < num__fg9 <= 0.28         0.009357
               num__fg5 <= -0.33         0.009257
              num__fg17 <= -0.57         0.008954
               num__fg1

In [23]:
final_results_path = ARTIFACT_DIR / "final_model_results_2015_rebuild.csv"
final_results_df.to_csv(final_results_path, index=False)

run_manifest = {
    "data_path": str(data_path),
    "target_col": target_col,
    "target_defined_from_bp": bool(TARGET_DEFINED_FROM_BP),
    "bp_leakage_guard_active": bool(BP_LEAKAGE_GUARD_ACTIVE),
    "bp_dropped_columns": BP_DROPPED_COLUMNS,
    "models_trained": list(final_models.keys()),
    "stage1_trials_per_model": STAGE1_TRIALS_PER_MODEL,
    "stage2_refinements_per_top_config": STAGE2_REFINEMENTS_PER_TOP_CONFIG,
    "top_k_stage1": TOP_K_STAGE1,
    "top_k_stage2": TOP_K_STAGE2,
    "epochs": {"stage1": STAGE1_EPOCHS, "stage2": STAGE2_EPOCHS, "final": FINAL_EPOCHS},
    "cv_folds": {"stage1": CV_FOLDS_STAGE1, "stage2": CV_FOLDS_STAGE2, "stack_oof": STACK_OOF_FOLDS},
    "stack_search": {
        "meta_trials": STACK_META_TRIALS,
        "blend_trials": STACK_BLEND_TRIALS,
        "top_base_models": STACK_TOP_BASE_MODELS,
    },
    "best_stack_strategy": stack_strategy if "stack_strategy" in globals() else None,
    "best_stack_calibration": best_stack_method if "best_stack_method" in globals() else None,
    "collinearity_cutoff": COLLINEARITY_CUTOFF,
    "final_feature_count": int(X_train_final.shape[1]),
    "best_model": best_model_name,
    "best_calibration": best_method,
}
manifest_path = ARTIFACT_DIR / "run_manifest_2015_rebuild.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print("Rebuild complete. Key artifacts:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name}")

print("\nModel files:")
for p in sorted(MODEL_DIR.glob("*.joblib")):
    print(f"- {p.name}")

Rebuild complete. Key artifacts:
- best_calibrated_summary_2015_rebuild.json
- calibration_results_2015_rebuild.csv
- final_candidate_selection_2015_rebuild.csv
- final_model_results_2015_rebuild.csv
- raw_variable_drop_audit_2015_rebuild.csv
- raw_variable_kept_identifier_like_2015_rebuild.csv
- run_manifest_2015_rebuild.json
- runtime_variable_list_2015_rebuild.csv
- runtime_variable_names_2015_rebuild.csv
- stack_calibration_results_2015_rebuild.csv
- stack_search_results_2015_rebuild.csv
- variable_audit_2015_rebuild.csv
- variable_audit_suspect_kept_2015_rebuild.csv
- variable_list_2015_rebuild.csv
- variable_list_suspect_final_2015_rebuild.csv

Model files:
- adaboost.joblib
- catboost.joblib
- knn.joblib
- lightgbm.joblib
- log_reg.joblib
- naive_bayes.joblib
- random_forest.joblib
- xgboost.joblib


In [24]:
# Fresh variable/identifier audit (independent cell, dictionary-aware).
def is_id_like_var(name):
    n = str(name).lower()
    return any(tok in n for tok in ["id", "subject", "participant", "seqn", "identifier", "record", "uuid", "case", "sample", "member_code", "hhnum"] )


def is_bp_like_var(name):
    n = str(name).lower()
    return any(tok in n for tok in ["ave_sbp", "ave_dbp", "sbp", "dbp", "systolic", "diastolic", "sysbp", "diabp", "blood_pressure"] )


dict_drop_set = set(globals().get("DICT_IDENTIFIER_DROPPED_COLUMNS", []))
base_cols = [c for c in df.columns if c != target_col]
train_base = X.loc[y_train.index]
train_uniq_ratio = {c: train_base[c].nunique(dropna=False) / max(len(train_base), 1) for c in train_base.columns}

rows = []
for col in base_cols:
    rows.append(
        {
            "stage": "raw_base",
            "variable": col,
            "identifier_like": is_id_like_var(col),
            "bp_like": is_bp_like_var(col),
            "train_nunique_ratio": train_uniq_ratio.get(col, np.nan),
            "dropped_bp_guard": col in BP_DROPPED_COLUMNS,
            "dropped_dict_rule": col in dict_drop_set,
            "dropped_id_rule": col in drop_id_like,
            "kept_after_base_filters": col in X_train.columns,
        }
    )

for col in sorted(set(X_train_fe.columns) - set(X_train.columns)):
    rows.append(
        {
            "stage": "engineered",
            "variable": col,
            "identifier_like": is_id_like_var(col),
            "bp_like": is_bp_like_var(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": True,
        }
    )

for col in X_train_final.columns:
    rows.append(
        {
            "stage": "final_model",
            "variable": col,
            "identifier_like": is_id_like_var(col),
            "bp_like": is_bp_like_var(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": True,
        }
    )

var_list_df = pd.DataFrame(rows).sort_values(["stage", "variable"]).reset_index(drop=True)
final_suspect_df = var_list_df[(var_list_df["stage"] == "final_model") & (var_list_df["identifier_like"] | var_list_df["bp_like"])].copy()
runtime_var_names_df = pd.DataFrame({"name": sorted([n for n in globals().keys() if not n.startswith("_")])})

var_list_path = ARTIFACT_DIR / "variable_list_2015_rebuild.csv"
suspect_path = ARTIFACT_DIR / "variable_list_suspect_final_2015_rebuild.csv"
runtime_names_path = ARTIFACT_DIR / "runtime_variable_names_2015_rebuild.csv"

var_list_df.to_csv(var_list_path, index=False)
final_suspect_df.to_csv(suspect_path, index=False)
runtime_var_names_df.to_csv(runtime_names_path, index=False)

print(f"Saved: {var_list_path}")
print(f"Saved: {suspect_path}")
print(f"Saved: {runtime_names_path}")
print(f"Dictionary files scanned: {len(globals().get('DICTIONARY_FILES_USED', []))}")
print(f"Dictionary ID columns dropped: {sorted(list(dict_drop_set))}")
print(var_list_df.groupby("stage")["variable"].count().to_string())
print(f"Potential id/bp-like final features: {len(final_suspect_df)}")
final_suspect_df.head(30)

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\variable_list_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\variable_list_suspect_final_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_retraining_artifacts\runtime_variable_names_2015_rebuild.csv
Dictionary files scanned: 10
Dictionary ID columns dropped: ['hhnum', 'member_code']
stage
engineered      1
final_model    29
raw_base       63
Potential id/bp-like final features: 0


,stage,variable,identifier_like,bp_like,train_nunique_ratio,dropped_bp_guard,dropped_dict_rule,dropped_id_rule,kept_after_base_filters
